# 설정 B 실험 전체 실행 (한 파일)

`run_setting_b_server.ipynb`(실행 안내)와 `m_setting_b_model.ipynb`(최종 모델 블렌드), 후속 실험 스크립트(`blend_extras.py`, `blend_tuned_test.py`)를 **노트북 하나로 합친 것**이다. 위에서부터 `모두 실행`하면 끝까지 돈다.

**준비물**: 팀 저장소의 `feature/setting-b-fe` 브랜치를 받은 폴더(`common.py`, `graph_features.py`, `feature_engineering.py`, `fe_experiment.py`, `forward_cv.py`가 있는 곳)에 이 노트북 하나만 두고 연다. 블렌드 노트북 파일은 따로 필요 없다.

| 단계 | 내용 | 시간 (12코어 기준) |
|---|---|---|
| 0 | 환경·데이터 확인 (없으면 자동 다운로드) | 1분 |
| 1 | E1~E3 설정 A·B (기본 꺼짐) | 10~20분 |
| 2 | 피처 가공 (`fe_experiment.py`) | 5분 |
| 3 | 전진 검증 (`forward_cv.py`) | 2분 |
| 4 | **최종 모델: 블렌드** (팀원 노트북 원문 그대로) | 7분 |
| 5 | 후속 실험: 튜닝·불균형·임계값·self-training·SHAP·오류 분석 | 16분 |
| 6 | 튜닝 블렌드 테스트 채점 (재현) | 3분 |
| 7 | 요약 | – |

- **테스트 규칙**: 4단계와 6단계의 테스트 채점은 이미 고정한 설정을 같은 방식으로 다시 채점하는 재현이다. 결과를 보고 설정을 바꾸지 않는다.
- 4~6단계는 같은 커널에서 이어서 돈다. 4단계를 건너뛰고 5·6단계만 돌릴 수는 없다.

In [ ]:
# ===== 실행 설정 =====
RUN_BASELINES = False      # 1단계 E1~E3 (결과가 이미 있으면 False)
RUN_FE = True              # 2단계 피처 가공
RUN_FORWARD_CV = True      # 3단계 전진 검증
N_JOBS_OVERRIDE = None     # 4~6단계 병렬 작업 수. None이면 CPU 코어 수 - 1

In [ ]:
import os, subprocess, sys, time, json, platform
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
assert (ROOT / "common.py").exists(), f"저장소 루트에서 열어야 한다: {ROOT}"
ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1", "MPLBACKEND": "Agg"}
pd.set_option("display.width", 220, "display.max_columns", 30)


def run(*args, timeout=None):
    """명령을 실행하며 출력을 실시간으로 보여 준다. 실패하면 멈춘다."""
    t0 = time.time()
    print("$", " ".join(map(str, args)), flush=True)
    p = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=ENV, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        if line.strip():
            print(line, end="", flush=True)
    p.wait(timeout=timeout)
    print(f"--- 종료 코드 {p.returncode}, {(time.time() - t0) / 60:.1f}분", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"실패: {' '.join(map(str, args))}")


def show_csv(path, n=30, cols=None):
    df = pd.read_csv(ROOT / path, encoding="utf-8-sig")
    display(df[cols] if cols else df.head(n))
    return df

## 0. 환경과 데이터 확인

In [ ]:
import importlib
print("Python", platform.python_version(), "| CPU", os.cpu_count(), "| 위치", ROOT)
need = {"numpy": "2.3.5", "pandas": "2.3.3", "sklearn": "1.7.2", "lightgbm": "4.7.0", "matplotlib": "3.10.7",
        "networkx": "3.5", "scipy": None, "joblib": None, "nbconvert": None, "ipykernel": None}
for mod, ver in need.items():
    try:
        v = importlib.import_module(mod).__version__
        print(f"  {mod:11s} {v:10s}" + ("" if ver is None or v == ver else f"  (팀 기준 {ver})"))
    except ImportError:
        print(f"  {mod:11s} 없음  → pip install {'scikit-learn' if mod == 'sklearn' else mod}")

data = ROOT / "data" / "elliptic"
files = ["elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"]
missing = [f for f in files if not (data / f).exists()]
print("데이터:", "있음" if not missing else f"없음 {missing}")

In [ ]:
# 데이터가 없을 때만 받는다 (약 680MB). Kaggle 토큰이 없으면 PyG 공개 미러에서 받는다.
if missing:
    run(sys.executable, "download_elliptic.py")
if not (ROOT / "feature_semantics.csv").exists():
    run(sys.executable, "build_feature_semantics.py")     # 인터넷 필요
print("준비 완료")

In [ ]:
# 그림의 한글 글꼴: 스크립트는 Windows의 'Malgun Gothic'을 쓴다. 서버(리눅스)에 없으면 글자가 네모로 나오지만
# 숫자 결과에는 영향이 없다. 나눔고딕이 있으면 그것으로 바꿔 쓴다.
from matplotlib import font_manager
fonts = {f.name for f in font_manager.fontManager.ttflist}
print("한글 글꼴:", [f for f in ["Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"] if f in fonts] or "없음")

## 1. E1~E3 (설정 A·B)

E1이 E2·E3 결과를 참고치로 읽어서 E2 → E3 → E1 순서로 돌린다.

In [ ]:
if RUN_BASELINES:
    for s in ["e2_baseline.py", "e3_graph_features.py", "e1_rules.py"]:
        run(sys.executable, s)
show_csv("results/e1_summary.csv", 15)
show_csv("results/e3_summary.csv", 15);

## 2. 피처 가공 (`fe_experiment.py`)

거래 모양, 선행 거래 모양, 직전 시기 백분위. 검증 구간 하나(31~34)로 고른 결과라 참고용이다.

In [ ]:
if RUN_FE:
    run(sys.executable, "fe_experiment.py")
show_csv("results/fe_summary.csv");

## 3. 전진 검증 (`forward_cv.py`)

학습 끝 T ∈ {14, 18, 22, 26, 30}, 예측 거리 1~4 / 5~8 / 9~15 step의 PR-AUC 평균으로 고른다. LightGBM 기본 0.677, 규칙 0.444가 나오면 블렌드 노트북과 같은 틀이다.

In [ ]:
if RUN_FORWARD_CV:
    run(sys.executable, "forward_cv.py")
show_csv("results/forward_cv_summary.csv", 12)
show_csv("results/forward_cv_test.csv")
display(Image(ROOT / "figures" / "forward_cv_horizon.png"))

---
# 4. 최종 모델: 블렌드 (`m_setting_b_model.ipynb` 원문)

아래부터 7단계 전까지는 팀원 노트북을 **내용 그대로** 옮겼다. 바꾼 것은 병렬 작업 수(`N_JOBS`) 한 줄뿐이다.

## 설정 B 모델 개선: 거래 시점 정보만으로 만든 새 모델

**무엇을 하는 노트북인가.** 팀의 설정 B(거래가 일어난 순간에 알 수 있는 정보만 사용)에서, 테스트 정보 유출 없이 성능을 올린 새 모델을 만들고 그 과정을 처음부터 끝까지 보여 준다.

**결과 요약** (테스트 35~49, 설정 B. 새 모델의 설정과 임계값은 테스트 전에 고정했다)

| 모델 | F1 | Precision | Recall | PR-AUC | F1 35~42 | F1 43~49 |
|---|---|---|---|---|---|---|
| 규칙: 입력 1개·출력 1개 (팀 깊이 2 트리와 같은 규칙) | **0.709** | 0.823 | 0.623 | 0.538 | 0.808 | 0.000 |
| LightGBM 기본 (팀 E2) | 0.545 | 0.594 | 0.504 | 0.585 | 0.668 | 0.085 |
| LightGBM 기본 + 팀 그래프 피처 (팀 E3) | 0.569 | 0.665 | 0.498 | 0.606 | 0.694 | 0.044 |
| **새 모델: 스코어카드 + LightGBM 블렌드** | 0.488 | 0.377 | 0.690 | **0.641** | 0.624 | 0.026 |

- **순위 품질(PR-AUC)은 올랐다.** 팀 최고 0.606 → 0.641. 폐쇄 전 구간(35~42)에서는 0.750 → 0.783이다.
- **F1은 오르지 않았다.** 검증에서 고정한 임계값(0.32)에서 F1은 0.488, Precision은 0.377이다. 임계값을 테스트 라벨로 골라도 F1은 0.634까지여서, 규칙(0.709)을 넘지 못한다.
- **이유**: 테스트 35~42 구간에서 "입력 1개·출력 1개" 거래의 불법 비율이 89.3%로 올랐다(학습 70.1%, 검증 66.3%). 규칙이 가장 잘 맞는 시기였다. 반대로 "출력 2개" 거래의 불법 비율은 검증 10.2%에서 테스트 35~42 3.0%로 내려갔다. 새 모델이 이 유형에서 더 잡으려 한 것은 대부분 오탐이 됐다(불법 판정 753건 중 14.6%만 불법).
- **43~49 구간은 새 모델도 실패한다** (F1 0.026). 이 구간에서는 "입력 1개·출력 1개" 거래의 불법 비율이 0.0%다.
- 테스트를 본 뒤에는 모델과 임계값을 바꾸지 않았다. 위 수치가 최종이다.

**과정에서 확인한 것**

- 금액·개수·블록 높이를 실제 단위로 되돌릴 수 있다(사토시 단위까지 일치). 절대 블록 높이는 코인 나이로, 절대 수수료는 시장 중앙값 대비 값으로 바꿨다.
- 라벨 없는 점검으로 "시점에 따라 수준이 변하는 피처" 30개를 걸렀다. 이런 피처는 1~4 step 앞에서만 이득이고 5 step 이후로는 손해다(5장).
- 모델 종류에 따른 차이는 작고, 2단 스태킹은 단순 평균보다 낫지 않다(6~7장).
- 재학습하면 일부 회복된다. 라벨이 2주 뒤 도착한다고 보면 PR-AUC 0.641 → 0.722, 43~49에서 잡는 불법이 9건 → 71건이다(10.2).
- 같은 종류의 라벨은 더 모아도 늘지 않는다. 학습 라벨의 10%만 쓴 검증 PR-AUC는 0.755, 전부 쓴 값은 0.779이다(10.3).

**읽는 순서**

| 장 | 내용 |
|---|---|
| 0 | 규칙과 준비 (팀 분할 확인) |
| 1 | 피처를 실제 단위로 되돌리기 (BTC, 개수, 블록 수) |
| 2 | 라벨 없이 하는 안정성 점검 (시간에 따라 수준이 변하는 피처 찾기) |
| 3 | 검증 방법 (1~34 안의 전진 검증) |
| 4 | 기준 모델을 검증 점수로 다시 재기 |
| 5 | 피처셋 비교: 예측 거리별로 보면 무엇이 버티는가 |
| 6 | 모델 종류 비교 |
| 7 | 블렌딩과 2단 스태킹 |
| 8 | 최종 설정 고정, 공식 검증(31~34) 점수 |
| 9 | 테스트(35~49) 채점: 한 번 |
| 10 | 사후 분석 (선택에 쓰지 않음): 오류 유형, 재학습 회복, 데이터가 더 필요한가, 무엇을 보고 판단하나 |
| 11 | 모델 저장과 사용법, 결론 |

### 0. 규칙과 준비

- **분할은 팀 `common.py` 그대로**: 학습 1~30 / 검증 31~34 / 테스트 35~49 (라벨 있는 거래만).
- **쓰는 정보는 설정 B 그대로**: 거래 자체 피처 `TX` 21개 + 선행 거래 집계 `NBR_IN` 36개 + 과거 쪽 연결(선행 거래). 주소 누적 통계 `ADDR`, 후속 거래 집계 `NBR_OUT`, out-degree, 라벨 기반 피처는 쓰지 않는다. 새로 만든 피처는 모두 이 정보를 **가공**한 것이다.
- **테스트는 한 번만 채점한다.** 피처 선택, 모델 종류, 하이퍼파라미터, 블렌딩 방법, 임계값은 9장 전에 1~34 구간만으로 정하고 고정한다.
- 팀 문서에 이미 있는 테스트 결과(얕은 트리 0.709, LightGBM 0.545 등)는 알고 시작했다. 이 지식이 "단순하고 안정적인 피처를 쓰자"는 방향 설정에 영향을 줬다는 점은 밝혀 둔다. 새 모델의 선택에는 테스트 점수를 쓰지 않았다.

In [ ]:
import hashlib
import json
import os
import time
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"      # 병렬 작업 프로세스의 경고도 끈다

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from joblib import Parallel, delayed
from matplotlib import font_manager
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.tree import DecisionTreeClassifier

from common import (BASE, DARK_MARKET_STEP, NBR_IN, TEST_STEPS, TRAIN_STEPS, TX, VAL_STEPS,
                    evaluate, load_dataset, load_edges, z_to_count)
from common import tune_threshold as team_tune_threshold
from graph_features import build_past_graph_features

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 70)
pd.options.display.float_format = "{:.3f}".format

N_JOBS = N_JOBS_OVERRIDE or max(1, (os.cpu_count() or 2) - 1)   # 서버 코어 수에 맞춤
SEEDS = (0, 1, 2)               # 검증용 시드
FINAL_SEEDS = (0, 1, 2, 3, 4)   # 최종 LightGBM 시드 (평균)
RES, FIG = BASE / "results", BASE / "figures"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
T_START = time.time()

C_NEW, C_TREE, C_LGBM, C_RULE, C_MUTE = "#2a78d6", "#eb6834", "#1baf7a", "#8a8f98", "#c3c7cf"


def use_korean_font():
    for p in [BASE.parent / "assets" / "fonts" / "NanumGothic-Regular.ttf", BASE / "NanumGothic-Regular.ttf"]:
        if p.exists():
            font_manager.fontManager.addfont(str(p))
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]:
        if name in have:
            plt.rcParams.update({"font.family": name, "axes.unicode_minus": False})
            return name
    return None


print("그림 글꼴:", use_korean_font())
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e8ec", "axes.axisbelow": True, "figure.dpi": 110})

In [ ]:
df_all = load_dataset(labeled_only=False)      # 203,769건. unknown도 그래프와 창 통계 계산에 쓴다(라벨은 쓰지 않는다)
edges = load_edges()
is_lab = df_all["y"].notna().values

rows = []
for name, steps in [("학습", TRAIN_STEPS), ("검증", VAL_STEPS), ("테스트", TEST_STEPS)]:
    d = df_all[is_lab & df_all["time_step"].isin(steps).values]
    rows.append({"구간": name, "time step": f"{steps.start}~{steps.stop - 1}", "거래 수": len(d),
                 "불법": int(d["y"].sum()), "합법": int((d["y"] == 0).sum()), "불법 비율": d["y"].mean()})
display(pd.DataFrame(rows))

test_ids = df_all.loc[is_lab & df_all["time_step"].isin(TEST_STEPS).values, "txId"]
sha = hashlib.sha256("\n".join(str(int(t)) for t in sorted(test_ids)).encode()).hexdigest()
print("테스트 거래 ID 목록 SHA-256:", sha)
assert sha == "ba5eb8159dc8eb20bce518f77c2363ba04eca3cb0faed04adce370771b3c63f4", "팀 테스트셋과 행이 다르다"

### 1. 피처를 실제 단위로 되돌리기

Elliptic 피처는 표준화(z-score)된 값이다. 피처 의미(Šafář et al., 2026)를 알면 원래 단위로 되돌릴 수 있다.

**개수.** 값의 간격이 일정하므로 팀의 `z_to_count`로 되돌린다(입력 수, 출력 수, 고유 주소 수).

**금액.** z 값과 원값은 직선 관계다. Elliptic++의 원값과 맞춰 구한 계수를 쓴다($R^2 = 1.000000$).

$$\text{BTC} = a_j \cdot z_j + b_j$$

**블록 높이와 코인 나이.** `agg_6, 7, 9`는 선행 거래(이 거래가 쓰는 코인을 만든 거래)의 블록 높이다. time step $s$의 첫 블록은 $391{,}200 + 2016\,(s-1)$이고, step별 `agg_7` 최댓값과 직선으로 맞는다(최대 잔차 6블록).

$$h = 30{,}705.62 \cdot z_{\texttt{agg\_7}} + 437{,}070.73, \qquad \text{age} = \underbrace{\max_{v \in \text{step } s} h_v}_{\text{그 시점의 현재 블록}} - h$$

절대 블록 높이는 "언제의 거래인가" 그 자체다. 학습 구간에서 본 적 없는 값이 테스트에 나오므로 그대로 쓰면 통하지 않는다. **현재 블록에서 뺀 코인 나이**는 어느 시기에나 같은 뜻을 가진다.

**시장 수준 대비 상대값.** 수수료와 금액은 같은 창(time step, 6~16블록)의 중앙값 대비 로그 차이로도 만든다. 배포할 때는 직전 몇 블록의 중앙값으로 대신하면 된다.

$$\text{rel}_x = \log_{10} x - \operatorname{median}_{v \in \text{step } s} \log_{10} x_v$$

In [ ]:
STEP0_BLOCK, STRIDE = 391_200, 2016
AFFINE = {   # z → BTC (기울기, 절편). 사토시 단위까지 맞도록 전체 정밀도로 적는다
    "local_0": (158.53961614979636, 27.424606039228657),      # 거래 금액
    "local_1": (0.003863237109353783, 0.0008134145375891326), # 수수료
    "local_7": (154.95285232436652, 25.35739216423343),       # 입력 금액 최솟값
    "local_8": (157.15537929625287, 26.63160914847197),       # 입력 금액 최댓값
    "local_10": (155.12596873124446, 25.74307707460844),      # 입력 금액 평균
    "local_15": (10.967286910576039, 0.6295355371031877),     # 출력 금액 최솟값
    "local_16": (154.2064249964282, 26.392485792178636),      # 출력 금액 최댓값
    "local_18": (66.95687375555653, 11.802101672840028),      # 출력 금액 평균
}
HEIGHT_AGG7 = (30705.62, 437070.73)        # agg_7 → 블록 높이


def level(z):
    """등간격 값을 0부터 시작하는 정수 수준으로 바꾼다."""
    u = np.unique(z)
    d = np.diff(u)
    return np.rint((z - u[0]) / d[d > 1e-9].min())


def safe_div(a, b):
    return np.divide(a, b, out=np.full(len(a), np.nan), where=np.abs(b) > 1e-12)


def chain_length(flag, pred):
    """flag가 참인 거래가 과거 방향(유일한 선행 거래)으로 몇 번 연속됐는지."""
    chain, has = flag.astype(int), pred >= 0
    for _ in range(len(flag)):
        prev = np.zeros_like(chain)
        prev[has] = chain[pred[has]]
        new = np.where(flag, 1 + prev, 0)
        if np.array_equal(new, chain):
            break
        chain = new
    return chain


def build_features(df, edges):
    """설정 B의 정보(TX, NBR_IN, 과거 쪽 연결)만으로 가공 피처를 만든다. 라벨은 쓰지 않는다."""
    n, f, step = len(df), {}, df["time_step"].values

    # ---- 개수와 거래 모양
    n_in, n_out = z_to_count(df["local_3"].values), z_to_count(df["local_4"].values)
    u_in, u_out = z_to_count(df["local_5"].values), z_to_count(df["local_13"].values)
    f["cnt_n_in"], f["cnt_n_out"], f["cnt_u_in"], f["cnt_u_out"] = n_in, n_out, u_in, u_out
    f["cnt_1in1out"] = ((n_in == 1) & (n_out == 1)).astype(int)
    f["cnt_1out"] = (n_out == 1).astype(int)
    f["cnt_2out"] = (n_out == 2).astype(int)
    f["cnt_out_minus_in"] = n_out - n_in
    f["cnt_log_out_in"] = np.log(n_out / n_in)
    f["cnt_in_addr_per_input"] = u_in / n_in
    f["cnt_out_addr_per_output"] = u_out / n_out
    est_size = 148.0 * n_in + 34.0 * n_out + 10.0                 # 바이트 (P2PKH 기준 추정)

    # ---- 금액 (BTC)
    dec = {c: a * df[c].values + b for c, (a, b) in AFFINE.items()}
    total, fee = dec["local_0"], dec["local_1"]
    in_min, in_max, in_mean = dec["local_7"], dec["local_8"], dec["local_10"]
    out_min, out_max, out_mean = dec["local_15"], dec["local_16"], dec["local_18"]
    in_total, out_total = in_mean * n_in, out_mean * n_out
    f["val_log_total"] = np.log10(np.clip(total, 1e-8, None))
    f["val_log_fee"] = np.log10(np.clip(fee, 1e-8, None))
    f["val_fee_frac"] = safe_div(fee, np.clip(in_total, 1e-8, None))
    f["val_feerate_est"] = fee * 1e8 / est_size                   # sat/byte (추정)
    f["val_fee_per_output"] = fee / n_out
    f["val_out_max_share"] = safe_div(out_max, out_total)
    f["val_out_min_share"] = safe_div(out_min, out_total)
    f["val_in_max_share"] = safe_div(in_max, in_total)
    f["val_in_min_share"] = safe_div(in_min, in_total)
    f["val_log_out_max"] = np.log10(np.clip(out_max, 1e-8, None))
    f["val_log_out_min"] = np.log10(np.clip(out_min, 1e-8, None))
    f["val_log_in_max"] = np.log10(np.clip(in_max, 1e-8, None))
    for name, v in [("total", total), ("out_max", out_max), ("out_min", out_min), ("fee", fee)]:
        cur = np.rint(np.clip(v, 0, None) * 1e8).astype(np.int64)
        tz = np.zeros(n, dtype=int)                               # 사토시 단위 끝자리 0의 개수 (딱 떨어지는 금액)
        for _ in range(8):
            more = (cur > 0) & (cur % 10 == 0)
            tz += more
            cur = np.where(more, cur // 10, cur)
        f[f"val_round_{name}"] = tz

    # ---- 코인 나이 (블록 수)
    h_max = np.rint(HEIGHT_AGG7[0] * df["agg_7"].values + HEIGHT_AGG7[1])
    single_in = n_in == 1          # 입력이 1개면 최솟값 = 최댓값 = 평균. 이 관계로 agg_6, agg_9의 눈금을 agg_7에 맞춘다
    h_min = np.rint(np.polyval(np.polyfit(df["agg_6"].values[single_in], h_max[single_in], 1), df["agg_6"].values))
    h_mean = np.polyval(np.polyfit(df["agg_9"].values[single_in], h_max[single_in], 1), df["agg_9"].values)
    h_min = np.minimum(h_min, h_max)
    ref = pd.Series(h_max).groupby(step).transform("max").values   # 그 step에서 관측된 가장 최근 블록
    f["age_young"] = np.clip(ref - h_max, 0, None)                 # 가장 최근에 받은 코인의 나이
    f["age_old"] = np.clip(ref - h_min, 0, None)
    f["age_mean"] = np.clip(ref - h_mean, 0, None)
    f["age_span"] = np.clip(h_max - h_min, 0, None)
    f["age_log_young"] = np.log10(1 + f["age_young"])
    f["age_log_old"] = np.log10(1 + f["age_old"])
    f["age_young_lt_window"] = (f["age_young"] <= 16).astype(int)  # 같은 창 안에서 받은 코인을 바로 씀
    f["age_young_lt_day"] = (f["age_young"] <= 144).astype(int)

    # ---- 시장 수준 대비 상대값 (rel_: 같은 창 중앙값, relp_: 직전 step 중앙값)
    levels = {"feerate": np.log10(np.clip(f["val_feerate_est"], 1e-3, None)), "fee": f["val_log_fee"],
              "total": f["val_log_total"], "out_max": f["val_log_out_max"], "out_min": f["val_log_out_min"],
              "in_max": f["val_log_in_max"]}
    steps_sorted = np.sort(np.unique(step))
    for name, v in levels.items():
        med = pd.Series(v).groupby(step).median()
        f[f"rel_{name}"] = v - med.reindex(step).values
        prev = med.shift(1).reindex(steps_sorted)
        prev.iloc[0] = med.iloc[0]
        f[f"relp_{name}"] = v - prev.reindex(step).values
    f["rel_feerate_rank"] = pd.Series(f["val_feerate_est"]).groupby(step).rank(pct=True).values
    f["rel_total_rank"] = pd.Series(total).groupby(step).rank(pct=True).values

    # ---- 과거 쪽 연결 (데이터셋 엣지의 선행 거래)
    pos = pd.Series(np.arange(n), index=df["txId"].values)
    src, dst = pos.loc[edges["txId1"]].values, pos.loc[edges["txId2"]].values
    P = sp.csr_matrix((np.ones(len(src)), (dst, src)), shape=(n, n))     # P[i, j] = 1: j가 i의 선행 거래
    in_deg = np.asarray(P.sum(axis=1)).ravel()
    Pn = sp.diags(np.divide(1.0, in_deg, out=np.zeros(n), where=in_deg > 0)) @ P
    pred = np.full(n, -1)
    single = np.flatnonzero(in_deg == 1)
    pred[single] = P.indices[P.indptr[single]]
    f["anc_in_share"] = np.clip(in_deg / n_in, 0, 1)                # 입력 중 같은 창 안 선행 거래에서 온 비중
    f["anc_has_parent"] = (in_deg > 0).astype(int)
    f["anc_chain_1in1out"] = chain_length((n_in == 1) & (n_out == 1), pred)
    f["anc_chain_1in"] = chain_length(n_in == 1, pred)
    depth = np.zeros(n)
    for _ in range(200):                                            # 창 안에서 가장 긴 과거 경로 길이
        new = np.where(in_deg > 0, 1 + np.asarray(P.multiply(depth[None, :]).max(axis=1).todense()).ravel(), 0)
        if np.array_equal(new, depth):
            break
        depth = new
    f["anc_depth"] = depth
    par = np.column_stack([f["age_young"], f["age_old"], f["cnt_1in1out"], f["cnt_1out"], n_out, n_in,
                           f["val_log_total"], f["val_feerate_est"], f["val_out_max_share"]])
    M1 = Pn @ np.nan_to_num(par)
    for j, name in enumerate(["age_young", "age_old", "is_1in1out", "is_1out", "n_out", "n_in", "log_total",
                              "feerate", "out_max_share"]):
        f[f"anc_p1_{name}"] = np.where(in_deg > 0, M1[:, j], np.nan)

    # ---- 선행 거래 집계(NBR_IN)를 같은 단위로 되돌려 자기 거래와 비교
    one = np.flatnonzero((n_in == 1) & (in_deg == 1))     # 입력 1개 + 그 선행 거래가 창 안에 있음 → 집계 = 그 거래의 값
    cal = {a: np.polyfit(df[a].values[one], df[l].values[pred[one]], 1) for a, l in [("agg_3", "local_0"), ("agg_15", "local_1")]}
    par_value = AFFINE["local_0"][0] * np.polyval(cal["agg_3"], df["agg_3"].values) + AFFINE["local_0"][1]
    par_fee = AFFINE["local_1"][0] * np.polyval(cal["agg_15"], df["agg_15"].values) + AFFINE["local_1"][1]
    f["par_log_value_mean"] = np.log10(np.clip(par_value, 1e-8, None))
    f["par_log_fee_mean"] = np.log10(np.clip(par_fee, 1e-8, None))
    f["par_value_ratio"] = np.log10(np.clip(total, 1e-8, None) / np.clip(par_value, 1e-8, None))
    f["par_fee_ratio"] = np.log10(np.clip(fee, 1e-8, None) / np.clip(par_fee, 1e-8, None))
    for name, v in [("par_value", f["par_log_value_mean"]), ("par_fee", f["par_log_fee_mean"])]:
        f[f"rel_{name}"] = v - pd.Series(v).groupby(step).median().reindex(step).values
    for name, c, off in [("min_n_in", "agg_24", 0), ("max_n_in", "agg_25", 0), ("min_n_out", "agg_30", 1), ("max_n_out", "agg_31", 1)]:
        f[f"par_{name}"] = level(df[c].values) + off      # 입력 수 = 수준(0 = 코인베이스), 출력 수 = 수준 + 1
    f["par_all_1out"] = (f["par_max_n_out"] == 1).astype(int)
    f["par_all_1in"] = ((f["par_max_n_in"] == 1) & (f["par_min_n_in"] == 1)).astype(int)
    f["par_coinbase"] = (f["par_min_n_in"] == 0).astype(int)        # 채굴 보상을 직접 쓰는 거래

    out = pd.DataFrame(f)
    out.insert(0, "txId", df["txId"].values)
    checks = {"total": total, "fee": fee, "n_in": n_in, "n_out": n_out, "in_deg": in_deg, "pred": pred}
    return out.merge(build_past_graph_features(df, edges), on="txId", how="left"), checks   # + 팀의 p_*, pin_*, p2in_*


t0 = time.time()
feats, chk = build_features(df_all, edges)
print(f"가공 피처 {feats.shape[1] - 1}개, {time.time() - t0:.0f}초")

In [ ]:
# 복호화가 맞는지 확인한다
for name in ["total", "fee"]:
    sat = chk[name] * 1e8
    share = np.mean(np.abs(sat - np.rint(sat)) < 0.01)
    print(f"{name}: 사토시 단위로 정수가 되는 비율 {share:.4f}")
    assert share > 0.999, "금액 복호화 계수가 맞지 않는다"

first_block = STEP0_BLOCK + STRIDE * (np.arange(1, 50) - 1)
zmax = df_all.groupby("time_step")["agg_7"].max().values
a, b = np.polyfit(zmax, first_block, 1)
print(f"블록 높이: {a:,.2f}·z + {b:,.2f}, step 첫 블록과의 최대 잔차 {np.abs(first_block - (a * zmax + b)).max():.1f}블록")

one = np.flatnonzero((chk["n_in"] == 1) & (chk["in_deg"] == 1))
ok_in = np.mean(level(df_all["agg_24"].values)[one] == chk["n_in"][chk["pred"][one]])
ok_out = np.mean(level(df_all["agg_30"].values)[one] + 1 == chk["n_out"][chk["pred"][one]])
print(f"선행 거래의 입력 수·출력 수 복호화: {len(one):,}건 중 일치 {ok_in:.4f} / {ok_out:.4f}")
print(f"코인 나이 중앙값 {np.median(feats['age_young']):.0f}블록, 1입력·1출력 거래 비율 {feats['cnt_1in1out'].mean():.3f}")

### 2. 라벨 없이 하는 안정성 점검

시간에 따라 **수준이 변하는 피처**는 학습 구간에서 잘 맞아도 미래 구간에서 뜻이 달라진다. 라벨 없이, 피처 값이 time step과 얼마나 같이 움직이는지만 재서 걸러낸다.

아래 그림은 전체 거래(unknown 포함)의 step별 분포다. 라벨은 쓰지 않았다. 수수료율 중앙값은 2016년 초 44 sat/B에서 2017년 중반 300 sat/B 이상으로 올랐고, 끝자리가 딱 떨어지는 고정 수수료는 거의 사라졌다. 반면 코인 나이와 거래 모양(1입력·1출력 비율)에는 추세가 없다.

In [ ]:
d = df_all[["txId", "time_step"]].merge(feats, on="txId")
g = d.groupby("time_step")
fig, axes = plt.subplots(2, 2, figsize=(11, 5.6), sharex=True)
panels = [("수수료율 중앙값 (sat/byte, 추정)", g["val_feerate_est"].median(), (g["val_feerate_est"].quantile(.25), g["val_feerate_est"].quantile(.75))),
          ("끝자리가 딱 떨어지는 수수료의 비율", g["val_round_fee"].apply(lambda s: (s >= 3).mean()), None),
          ("가장 최근 코인의 나이 중앙값 (블록)", g["age_young"].median(), None),
          ("1입력·1출력 거래의 비율", g["cnt_1in1out"].mean(), None)]
for ax, (title, s, band) in zip(axes.ravel(), panels):
    ax.axvspan(34.5, 49.5, color="#f0f2f5", zorder=0)
    ax.axvline(DARK_MARKET_STEP - 0.5, color="#6b7280", ls="--", lw=1)
    if band is not None:
        ax.fill_between(s.index, band[0].values, band[1].values, color=C_NEW, alpha=0.18, lw=0)
    ax.plot(s.index, s.values, color=C_NEW, lw=2)
    ax.set_title(title, loc="left", fontsize=10.5)
    ax.set_ylim(bottom=0)
for ax in axes[1]:
    ax.set_xlabel("time step  (회색 = 테스트 구간, 점선 = 43)")
fig.tight_layout()
fig.savefig(FIG / "m_drift_label_free.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
TREND_MAX, ETA2_MAX = 0.2, 0.15      # 넘으면 '시점에 따라 변함'

sem = pd.read_csv(BASE / "feature_semantics.csv").set_index("column")["description"]
scr = df_all.loc[df_all["time_step"] <= 34, ["txId", "time_step"] + TX + NBR_IN].merge(feats, on="txId")   # 1~34만. 테스트 구간은 보지 않는다
rows = []
for c in [c for c in scr.columns if c not in ("txId", "time_step")]:
    v = scr[c].astype(float)
    m = v.notna()
    vv, ss = v[m], scr["time_step"][m]
    gm = vv.groupby(ss).agg(["mean", "size"])
    tot = float(((vv - vv.mean()) ** 2).sum())
    eta2 = float((gm["size"] * (gm["mean"] - vv.mean()) ** 2).sum() / tot) if tot > 0 else 0.0
    trend = abs(float(vv.rank().corr(ss.rank()))) if vv.nunique() > 1 else 0.0
    rows.append({"피처": c, "추세 |ρ|": trend, "η²": eta2, "의미": sem.get(c, "")})
stab = pd.DataFrame(rows)
stab["시점에 따라 변함"] = (stab["추세 |ρ|"] > TREND_MAX) | (stab["η²"] > ETA2_MAX)
UNSTABLE = set(stab.loc[stab["시점에 따라 변함"], "피처"])
print(f"후보 {len(stab)}개 중 시점에 따라 변하는 피처 {len(UNSTABLE)}개")
display(stab[stab["시점에 따라 변함"]].sort_values("추세 |ρ|", ascending=False).drop(columns="시점에 따라 변함").reset_index(drop=True))

걸러진 피처는 세 부류다.

- **절대 블록 높이** (`agg_6, 7, 9`): 시점 그 자체. 코인 나이 `age_*`로 대체한다.
- **절대 수수료 수준** (`local_1`, `val_log_fee`, `val_feerate_est`, 선행 거래 수수료 `agg_12, 13, 15` 등): 시장 수준 대비 `rel_*`로 대체한다.
- **의미 미확인 피처 `local_2` 계열**과 **체인 길이 계열** (`p_peel_chain`, `anc_depth` 등): step마다 값이 크게 다르다. 팀 문서 6.4의 "peel chain은 학습 구간에서만 통했다"와 같은 결론이 라벨 없이도 나온다.

직전 step 중앙값을 쓴 `relp_*`도 걸러진다. 2주 전 수준과 지금 수준의 차이가 그대로 남기 때문이다.

In [ ]:
def cols_of(*prefixes):
    return [c for c in feats.columns if c.startswith(prefixes)]


RAW = TX + NBR_IN                                                       # 팀 설정 B 그대로 (57개)
ENG = cols_of("cnt_", "val_", "age_", "rel_", "par_", "anc_", "p_")     # 가공 피처 (relp_ 제외)
GRAPH = cols_of("pin_", "p2in_")                                        # 팀의 조상 거래 평균
keep = lambda cols: [c for c in cols if c not in UNSTABLE]

SETS = {
    "raw (팀 설정 B 57개)": RAW,
    "raw_stable (raw에서 변하는 피처 제외)": keep(RAW),
    "plus (raw + 가공 피처 전부)": RAW + ENG,
    "stable (안정 피처만)": keep(RAW + ENG + GRAPH),
}
print({k: len(v) for k, v in SETS.items()})

lab = df_all.loc[is_lab, ["txId", "time_step", "y"] + RAW].merge(feats, on="txId").reset_index(drop=True)
lab["y"] = lab["y"].astype(int)
dev = lab[lab["time_step"] <= 34].reset_index(drop=True)     # 개발용: 1~34. 9장 전까지는 이것만 쓴다
dev_f = dev.fillna(-1.0)                                     # 결측을 받지 않는 모델용 (선행 거래 없음 등 → -1)
print(f"개발 데이터 {len(dev):,}건 (불법 {dev['y'].sum():,})")

### 3. 검증 방법: 1~34 안의 전진 검증

팀의 공식 검증은 "1~30 학습 → 31~34 검증"이다. 이 구간은 불법이 508건이고 그중 342건이 step 32 한 곳에 몰려 있어서, 이것 하나만으로 고르면 선택이 흔들린다(단일 결정 트리 실험에서 확인). 그래서 같은 규칙을 과거로 반복한다.

$$\text{폴드 } T:\quad \text{학습 } \{t \le T\} \;\longrightarrow\; \text{검증 } \{T < t \le 34\}, \qquad T \in \{14, 18, 22, 26, 30\}$$

마지막 폴드($T=30$)의 다음 4 step이 팀의 공식 검증과 같다. 테스트는 학습 끝에서 1~15 step 떨어져 있으므로, 검증도 **예측 거리** $h = t - T$로 나눠 본다.

- **PR-AUC**: $h$ 1~4 / 5~8 / 9~15 구간별 값과 그 평균
- **F1**: 임계값을 그보다 과거 폴드들의 예측에서 정해 적용한 값. 테스트에서 임계값을 쓰는 방식(과거에서 정해 미래에 적용)을 그대로 흉내 낸다

$$\tau_T = \arg\max_{\tau} F_1\big(\text{과거 폴드들의 예측},\, \tau\big), \qquad F_1^{(T,h)} = F_1\big(\hat p_T \ge \tau_T\big)$$

In [ ]:
FOLD_CUTS, NEXT = [14, 18, 22, 26, 30], 4
BUCKETS = [(1, 4), (5, 8), (9, 15)]
Y, STEP = dev["y"].values, dev["time_step"].values
FOLDS = {T: {"train": STEP <= T, "next": (STEP > T) & (STEP <= T + NEXT), "long": STEP > T} for T in FOLD_CUTS}


def best_f1(y, prob):
    """라벨을 보고 고른 임계값의 F1 (도달 가능한 상한, 진단용)."""
    p, r, _ = precision_recall_curve(y, prob)
    return float((2 * p * r / np.clip(p + r, 1e-12, None)).max())


def tune_threshold(y, prob):
    """F1이 최대인 임계값. 후보는 예측값의 분위수라서 확률 눈금이 달라도 쓸 수 있다."""
    grid = np.unique(np.quantile(prob, np.linspace(0.30, 0.999, 500)))
    return float(grid[int(np.argmax([f1_score(y, prob >= t, zero_division=0) for t in grid]))])


def _fit_predict(make_model, X, y, tr, ev, seed):
    return make_model(seed).fit(X[tr], y[tr]).predict_proba(X[ev])[:, 1]


def run_cv(make_model, cols, seeds=SEEDS, fill=False):
    """폴드별 예측 {T: 배열}. 학습 행은 NaN, 시드 평균."""
    X = (dev_f if fill else dev)[cols].to_numpy(dtype=np.float32)
    jobs = [(T, s) for T in FOLD_CUTS for s in seeds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_predict)(make_model, X, Y, FOLDS[T]["train"], FOLDS[T]["long"], s) for T, s in jobs)
    preds = {T: np.full(len(dev), np.nan) for T in FOLD_CUTS}
    for T in FOLD_CUTS:
        preds[T][FOLDS[T]["long"]] = np.mean([p for (t, _), p in zip(jobs, res) if t == T], axis=0)
    return preds


def score(preds, folds=FOLD_CUTS):
    """예측 거리별 PR-AUC와, 과거에서 정한 임계값을 적용한 F1."""
    out = {}
    for lo, hi in BUCKETS:
        aps, f1s = [], []
        for T in folds:
            m = (STEP >= T + lo) & (STEP <= min(T + hi, 34))
            if m.sum() == 0 or Y[m].sum() == 0:
                continue
            aps.append(average_precision_score(Y[m], preds[T][m]))
            past = [t for t in folds if t < T]
            if past:
                th = tune_threshold(np.concatenate([Y[FOLDS[t]["next"]] for t in past]),
                                    np.concatenate([preds[t][FOLDS[t]["next"]] for t in past]))
                f1s.append(f1_score(Y[m], preds[T][m] >= th, zero_division=0))
        out[f"PR-AUC h{lo}~{hi}"] = float(np.mean(aps))
        out[f"F1 h{lo}~{hi}"] = float(np.mean(f1s))
    out["PR-AUC 평균"] = float(np.mean([out[f"PR-AUC h{lo}~{hi}"] for lo, hi in BUCKETS]))
    out["F1 평균"] = float(np.mean([out[f"F1 h{lo}~{hi}"] for lo, hi in BUCKETS]))
    v = FOLDS[30]["next"]
    out["공식 검증 PR-AUC"] = float(average_precision_score(Y[v], preds[30][v]))
    return out


def oof(preds):
    """각 폴드의 다음 4 step 예측을 이어 붙인 것 (step 15~34)."""
    out = np.full(len(dev), np.nan)
    for T in FOLD_CUTS:
        out[FOLDS[T]["next"]] = preds[T][FOLDS[T]["next"]]
    return out


def table(results, order="PR-AUC 평균"):
    cols = ["PR-AUC h1~4", "PR-AUC h5~8", "PR-AUC h9~15", "PR-AUC 평균", "F1 h1~4", "F1 h5~8", "F1 h9~15", "F1 평균", "공식 검증 PR-AUC"]
    t = pd.DataFrame(results).T[cols]
    return t.sort_values(order, ascending=False) if order else t

### 4. 기준 모델을 검증 점수로 다시 재기

팀 문서의 설정 B 기준 모델(규칙, 깊이 2 트리, 기본 LightGBM)을 같은 검증으로 잰다.

In [ ]:
class FlagRule:
    """0/1 열 하나를 그대로 점수로 내는 규칙."""
    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        s = X[:, 0].astype(float)
        return np.column_stack([1 - s, s])


def lgbm(n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=20, colsample_bytree=0.5, reg_lambda=0.0, **kw):
    def make(seed):
        return lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, num_leaves=num_leaves,
                                  min_child_samples=min_child_samples, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=colsample_bytree, reg_lambda=reg_lambda, random_state=seed,
                                  n_jobs=4, verbose=-1, **kw)
    return make


LGBM_DEFAULT = lgbm()                                                     # 팀 E2 설정
LGBM_MID = lgbm(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=40, reg_lambda=5.0)

P = {}       # 이름 → 폴드별 예측
t0 = time.time()
P["규칙: 입력 1개·출력 1개"] = run_cv(lambda s: FlagRule(), ["cnt_1in1out"], seeds=(0,))
P["규칙: 출력 1개"] = run_cv(lambda s: FlagRule(), ["cnt_1out"], seeds=(0,))
P["결정 트리 깊이 2 (raw)"] = run_cv(lambda s: DecisionTreeClassifier(max_depth=2, random_state=42), RAW, seeds=(0,))
P["LightGBM 기본 (raw)"] = run_cv(LGBM_DEFAULT, RAW)
base_names = list(P)
print(f"{time.time() - t0:.0f}초")
display(table({k: score(P[k]) for k in base_names}))

**검증에서는 기본 LightGBM이 규칙보다 훨씬 높다.** 그런데 팀 문서의 테스트 결과는 반대다(규칙·트리 F1 0.65~0.71, 기본 LightGBM 0.545). 1~34 안에서 잰 가까운 미래의 점수만 보면 이 역전이 보이지 않는다.

단서는 예측 거리다. 기본 LightGBM의 PR-AUC는 $h$가 커질수록 빠르게 떨어진다. 다음 장에서 이 하락이 어떤 피처에서 오는지 본다.

### 5. 피처셋 비교: 예측 거리별로 보면 무엇이 버티는가

같은 LightGBM(규제를 조금 준 설정: 잎 15, 트리 300)으로 피처셋만 바꾼다.

In [ ]:
t0 = time.time()
for name, cols in SETS.items():
    P[f"LightGBM 중간 | {name}"] = run_cv(LGBM_MID, cols)
print(f"{time.time() - t0:.0f}초")
fs_names = [f"LightGBM 중간 | {n}" for n in SETS]
fs_table = table({k: score(P[k]) for k in ["LightGBM 기본 (raw)"] + fs_names}, order=None)
display(fs_table)

fig, ax = plt.subplots(figsize=(7.6, 3.9))
xs = np.arange(len(BUCKETS))
styles = [("LightGBM 기본 (raw)", C_MUTE, "--"), (fs_names[0], C_RULE, "-"), (fs_names[2], C_TREE, "-"), (fs_names[1], C_LGBM, "-"), (fs_names[3], C_NEW, "-")]
for name, color, ls in styles:
    ys = [fs_table.loc[name, f"PR-AUC h{lo}~{hi}"] for lo, hi in BUCKETS]
    ax.plot(xs, ys, marker="o", color=color, ls=ls, lw=2, label=name.replace("LightGBM 중간 | ", ""))
    ax.annotate(f"{ys[-1]:.3f}", (xs[-1], ys[-1]), textcoords="offset points", xytext=(8, -3), fontsize=9, color=color)
ax.set_xticks(xs, [f"{lo}~{hi} step 앞" for lo, hi in BUCKETS])
ax.set_xlim(-0.2, 2.45)
ax.set_ylabel("검증 PR-AUC")
ax.set_title("예측 거리가 멀어질수록: 변하는 피처를 쓴 모델이 먼저 무너진다", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower left")
fig.savefig(FIG / "m_horizon.png", dpi=130, bbox_inches="tight")
plt.show()

- 절대 수수료와 블록 높이가 들어간 피처셋(`raw`, `plus`)은 **바로 다음 1~4 step**에서는 가장 높지만, 5 step 이후로는 안정 피처만 쓴 쪽보다 낮아진다.
- 테스트는 학습 끝에서 1~15 step 떨어져 있다. 먼 미래까지 버티는 쪽을 고르는 것이 맞다. 그래서 **안정 피처만 쓴다**(`stable`).
- 이 선택은 1~34 구간의 라벨과, 라벨 없는 안정성 점검만으로 했다.

### 6. 모델 종류 비교

안정 피처 위에서 모델 종류와 학습 방식을 바꾼다.

- **LightGBM 클래스 가중** (`class_weight="balanced"`, 팀 계획 E5의 불균형 처리)
- **Random Forest**, **ExtraTrees** (배깅)
- **로지스틱 스코어카드**: 각 피처를 분위수 8구간으로 나눠 one-hot으로 만든 뒤 로지스틱 회귀. 피처마다 구간별 점수를 더하는 구조라 해석이 쉽고, 상호작용을 외우지 못해서 과적합이 적다

$$\operatorname{logit} \hat p(\mathbf{x}) = \beta_0 + \sum_{j} \sum_{k=1}^{8} \beta_{jk}\, \mathbb{1}\big[x_j \in \text{구간}_{jk}\big]$$

In [ ]:
STABLE = SETS["stable (안정 피처만)"]
LGBM_CW = lgbm(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=40, reg_lambda=5.0, class_weight="balanced")


def make_scorecard(seed=0):
    return make_pipeline(KBinsDiscretizer(n_bins=8, encode="onehot", strategy="quantile", quantile_method="averaged_inverted_cdf"),
                         LogisticRegression(C=0.05, max_iter=3000))


t0 = time.time()
P["LightGBM 중간"] = P[f"LightGBM 중간 | stable (안정 피처만)"]
P["LightGBM 클래스 가중"] = run_cv(LGBM_CW, STABLE)
P["Random Forest"] = run_cv(lambda s: RandomForestClassifier(n_estimators=500, max_features="sqrt", min_samples_leaf=5, n_jobs=4, random_state=s), STABLE, fill=True)
P["ExtraTrees"] = run_cv(lambda s: ExtraTreesClassifier(n_estimators=500, max_features=0.3, min_samples_leaf=3, n_jobs=4, random_state=s), STABLE, fill=True)
P["스코어카드"] = run_cv(make_scorecard, STABLE, seeds=(0,), fill=True)
print(f"{time.time() - t0:.0f}초")
model_names = ["스코어카드", "LightGBM 클래스 가중", "LightGBM 중간", "ExtraTrees", "Random Forest"]
display(table({k: score(P[k]) for k in model_names}))

모델 종류 사이의 차이는 작다(PR-AUC 평균 0.74~0.77). 가장 단순한 스코어카드가 가장 높거나 그에 가깝다. 별도 스크립트(`m7_candidates.py`, `m9_sensitivity.py`)에서 CatBoost, MLP, 최근 step 가중, 설정 민감도까지 봤다. 결과는 `results/m7_candidates.csv`, `results/m9_sensitivity.csv`에 있다.

- CatBoost 0.766, MLP 0.747, 최근 가중 0.755: 뚜렷한 이득이 없다. CatBoost는 팀 `requirements.txt`에 없는 패키지라 넣지 않는다.
- 스코어카드는 구간 5~12개, 규제 C 0.02~0.2에서 0.753~0.768, LightGBM은 잎 7~31에서 0.767~0.772로 평탄하다. 설정을 더 조이지 않는다.

### 7. 블렌딩과 2단 스태킹

여러 모델을 섞는 방법 세 가지를 비교한다. 백분위와 스태킹은 과거 폴드의 예측이 있어야 하므로 $T \ge 18$ 폴드에서만 값이 나온다. 이 장의 표는 모두 같은 폴드끼리 비교한다.

- **평균**: 확률의 단순 평균
- **백분위**: 각 모델 점수를 과거 예측 분포에서의 백분위로 바꿔 평균 (배포 가능한 순위 평균)
- **스태킹(2단)**: 1단 모델들의 로짓을 입력으로 로지스틱 회귀를 한 번 더 학습. 2단 모델은 과거 폴드의 예측으로만 학습한다

$$\hat p_{\text{stack}} = \sigma\Big(w_0 + \sum_m w_m \operatorname{logit} \hat p_m\Big)$$

In [ ]:
EVAL_T = [18, 22, 26, 30]


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def combine(names, how):
    out = {}
    for T in EVAL_T:
        past = [t for t in FOLD_CUTS if t < T]
        if how == "평균":
            p = np.mean([P[n][T] for n in names], axis=0)
        elif how == "백분위":
            cols = []
            for n in names:
                ref = np.sort(np.concatenate([P[n][t][FOLDS[t]["next"]] for t in past]))
                cols.append(np.searchsorted(ref, P[n][T], side="right") / len(ref))
            p = np.mean(cols, axis=0)
        else:
            Xp = np.column_stack([np.concatenate([logit(P[n][t][FOLDS[t]["next"]]) for t in past]) for n in names])
            yp = np.concatenate([Y[FOLDS[t]["next"]] for t in past])
            meta = LogisticRegression(C=1.0, max_iter=1000).fit(Xp, yp)
            p = meta.predict_proba(np.column_stack([logit(np.where(np.isfinite(P[n][T]), P[n][T], 0.5)) for n in names]))[:, 1]
        out[T] = np.where(FOLDS[T]["long"], p, np.nan)
    return out


combos = {
    "스코어카드": ["스코어카드"],
    "LightGBM 클래스 가중": ["LightGBM 클래스 가중"],
    "스코어카드 + LightGBM": ["스코어카드", "LightGBM 클래스 가중"],
    "스코어카드 + LightGBM + ExtraTrees": ["스코어카드", "LightGBM 클래스 가중", "ExtraTrees"],
    "스코어카드 + LightGBM + ExtraTrees + RF": ["스코어카드", "LightGBM 클래스 가중", "ExtraTrees", "Random Forest"],
    "안정 피처 5개 모델 전부": model_names,
}
rows = {}
for name, members in combos.items():
    for how in (["평균"] if len(members) == 1 else ["평균", "백분위", "스태킹"]):
        rows[f"{name} | {how}"] = score(combine(members, how), folds=EVAL_T)
blend_table = table(rows)
display(blend_table[["PR-AUC h1~4", "PR-AUC h5~8", "PR-AUC h9~15", "PR-AUC 평균", "F1 평균", "공식 검증 PR-AUC"]])

- **섞으면 단일 모델보다 낫다.** 다만 이득은 PR-AUC 0.01 안팎이다.
- **2단 스태킹은 단순 평균·백분위보다 낫지 않다.** 2단 모델이 과거 폴드에 맞춘 가중치가 다음 구간으로 옮겨 가지 않는다.
- **모델을 3~5개로 늘려도 2개와 차이가 없다.** 표의 위쪽 조합들은 서로 잡음 범위 안에 있다.

### 8. 최종 설정 고정, 공식 검증 점수

여기까지의 검증 결과로 최종 설정을 정하고, **이 아래로는 바꾸지 않는다.**

| 항목 | 고정한 값 | 근거 |
|---|---|---|
| 피처 | 안정 피처 (`STABLE`) | 5장: 5 step 이후로 가장 잘 버틴다 |
| 모델 1 | 스코어카드 (분위수 8구간 + 로지스틱, C = 0.05) | 6장: 단일 모델 중 최상위, 설정에 둔감 |
| 모델 2 | LightGBM 클래스 가중 (트리 300, 잎 15), 시드 5개 평균 | 6장: 비선형 모델 중 최상위 |
| 결합 | 두 확률의 평균 | 7장: 가장 단순하고, 더 복잡한 방법과 차이 없음 |
| 임계값 | 블렌드의 전진 검증 예측(step 15~34)에서 F1 최대 | 3장 |

엄격한 변형도 하나 같이 잰다. `rel_*`(같은 창 중앙값 대비)를 `relp_*`(직전 step 중앙값 대비)로 바꾼 것으로, 같은 창의 다른 거래를 전혀 보지 않는다. 선택에는 쓰지 않고 결과만 함께 보고한다.

In [ ]:
FINAL_COLS = STABLE
STRICT_COLS = [c for c in STABLE if not c.startswith("rel_")] + cols_of("relp_")

P["블렌드"] = {T: (P["스코어카드"][T] + P["LightGBM 클래스 가중"][T]) / 2 for T in FOLD_CUTS}
ps, pl = run_cv(make_scorecard, STRICT_COLS, seeds=(0,), fill=True), run_cv(LGBM_CW, STRICT_COLS)
P["블렌드 (엄격한 과거 기준)"] = {T: (ps[T] + pl[T]) / 2 for T in FOLD_CUTS}


def oof_threshold(name, upto=34):
    o = oof(P[name])
    m = np.isfinite(o) & (STEP <= upto)
    return tune_threshold(Y[m], o[m])


THRESHOLD = oof_threshold("블렌드")
THRESHOLD_STRICT = oof_threshold("블렌드 (엄격한 과거 기준)")
THRESHOLD_SC, THRESHOLD_LG = oof_threshold("스코어카드"), oof_threshold("LightGBM 클래스 가중")
print(f"안정 피처 {len(FINAL_COLS)}개 | 임계값: 블렌드 {THRESHOLD:.3f}, 엄격한 변형 {THRESHOLD_STRICT:.3f}")
display(table({k: score(P[k]) for k in ["블렌드", "블렌드 (엄격한 과거 기준)", "스코어카드", "LightGBM 클래스 가중", "LightGBM 기본 (raw)", "규칙: 입력 1개·출력 1개"]}))

In [ ]:
# 팀 공식 검증: 학습 1~30 → 검증 31~34. 임계값은 검증보다 과거(step 15~30)의 예측에서 정한다.
v = FOLDS[30]["next"]
tr30 = FOLDS[30]["train"]
rows = []


def val_row(name, prob, thr, note):
    e = evaluate(Y[v], prob, thr)
    rows.append({"모델": name, "임계값": note, "Precision": e["precision"], "Recall": e["recall"], "F1": e["f1"], "PR-AUC": e["pr_auc"]})


val_row("규칙: 입력 1개·출력 1개", P["규칙: 입력 1개·출력 1개"][30][v], 0.5, "규칙")
val_row("규칙: 출력 1개", P["규칙: 출력 1개"][30][v], 0.5, "규칙")
tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(dev.loc[tr30, RAW], Y[tr30])
th_tree = team_tune_threshold(Y[tr30], tree.predict_proba(dev.loc[tr30, RAW])[:, 1])
val_row("결정 트리 깊이 2 (raw)", tree.predict_proba(dev.loc[v, RAW])[:, 1], th_tree, "학습 구간 F1 최대")
val_row("LightGBM 기본 (raw)", P["LightGBM 기본 (raw)"][30][v], 0.5, "0.5")
for name in ["스코어카드", "LightGBM 클래스 가중", "블렌드", "블렌드 (엄격한 과거 기준)"]:
    val_row(name, P[name][30][v], oof_threshold(name, upto=30), "step 15~30 예측에서 F1 최대")
val_table = pd.DataFrame(rows).set_index("모델")
display(val_table)

**공식 검증만 보면 기본 LightGBM(raw)의 F1이 가장 높다.** 이 검증은 학습 끝에서 1~4 step 앞만 본다. 5장에서 봤듯 raw 피처의 이득은 그 구간에 한정되고, 테스트는 1~15 step 앞이다. 그래서 선택 기준을 처음부터 예측 거리 평균으로 잡았고, 그 기준에서는 블렌드가 가장 높다(위 표의 PR-AUC 평균). 이 판단이 맞았는지는 다음 장의 테스트가 답한다.

### 9. 테스트(35~49) 채점

고정한 설정으로 1~34 전체를 다시 학습하고 테스트를 한 번 채점한다. 팀 기준 모델도 같은 자리에서 다시 계산해 나란히 놓는다.

In [ ]:
train_all = lab[lab["time_step"] <= 34].reset_index(drop=True)
test = lab[lab["time_step"] >= 35].reset_index(drop=True)
y_te, s_te = test["y"].values, test["time_step"].values
PERIODS = {"35~49": np.ones(len(test), bool), "35~42": s_te < DARK_MARKET_STEP, "43~49": s_te >= DARK_MARKET_STEP}


def fit_blend(train_df, cols, seeds=FINAL_SEEDS):
    """최종 설정의 두 모델을 학습한다. 반환: predict(df) → (블렌드, 스코어카드, LightGBM) 확률, 학습된 모델."""
    ytr = train_df["y"].values
    sc = make_scorecard().fit(train_df[cols].fillna(-1.0).to_numpy(np.float32), ytr)
    lgs = [LGBM_CW(s).fit(train_df[cols].to_numpy(np.float32), ytr) for s in seeds]

    def predict(d):
        p_sc = sc.predict_proba(d[cols].fillna(-1.0).to_numpy(np.float32))[:, 1]
        p_lg = np.mean([m.predict_proba(d[cols].to_numpy(np.float32))[:, 1] for m in lgs], axis=0)
        return (p_sc + p_lg) / 2, p_sc, p_lg
    return predict, {"scorecard": sc, "lgbm": lgs}


def test_metrics(prob, thr):
    out = {}
    for name, m in PERIODS.items():
        e = evaluate(y_te[m], prob[m], thr)
        out[name] = {"F1": e["f1"], "Precision": e["precision"], "Recall": e["recall"], "PR-AUC": e["pr_auc"],
                     "ROC-AUC": roc_auc_score(y_te[m], prob[m]), "사후 최적 F1": best_f1(y_te[m], prob[m])}
    return out


def flat(name, setting, note, mets):
    """여러 시드면 지표를 시드 평균한다(팀 방식)."""
    avg = {p: {k: float(np.mean([m[p][k] for m in mets])) for k in mets[0][p]} for p in PERIODS}
    return {"모델": name, "피처": setting, "임계값": note, "F1": avg["35~49"]["F1"], "Precision": avg["35~49"]["Precision"],
            "Recall": avg["35~49"]["Recall"], "PR-AUC": avg["35~49"]["PR-AUC"], "F1 35~42": avg["35~42"]["F1"],
            "F1 43~49": avg["43~49"]["F1"], "PR-AUC 35~42": avg["35~42"]["PR-AUC"], "PR-AUC 43~49": avg["43~49"]["PR-AUC"],
            "ROC-AUC 43~49": avg["43~49"]["ROC-AUC"], "사후 최적 F1": avg["35~49"]["사후 최적 F1"]}


t0 = time.time()
TEST_PROB, results = {}, []

# ---- 팀 기준 모델 (설정 B)
for name, col in [("규칙: 입력 1개·출력 1개", "cnt_1in1out"), ("규칙: 출력 1개", "cnt_1out")]:
    TEST_PROB[name] = test[col].values.astype(float)
    results.append(flat(name, "설정 B", "규칙", [test_metrics(TEST_PROB[name], 0.5)]))
tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(train_all[RAW], train_all["y"])
th_tree = team_tune_threshold(train_all["y"].values, tree.predict_proba(train_all[RAW])[:, 1])
TEST_PROB["결정 트리 깊이 2"] = tree.predict_proba(test[RAW])[:, 1]
results.append(flat("결정 트리 깊이 2", "설정 B raw 57", "학습 구간 F1 최대", [test_metrics(TEST_PROB["결정 트리 깊이 2"], th_tree)]))
TEAM_GRAPH = RAW + cols_of("p_", "pin_", "p2in_")
for name, cols in [("LightGBM 기본", RAW), ("LightGBM 기본 + 팀 그래프 피처", TEAM_GRAPH)]:
    probs = [LGBM_DEFAULT(s).fit(train_all[cols], train_all["y"]).predict_proba(test[cols])[:, 1] for s in SEEDS]
    TEST_PROB[name] = np.mean(probs, axis=0)
    results.append(flat(name, f"설정 B {len(cols)}개", "0.5", [test_metrics(p, 0.5) for p in probs]))

# ---- 새 모델 (고정한 설정)
predict_final, FINAL_MODELS = fit_blend(train_all, FINAL_COLS)
TEST_PROB["새 모델: 블렌드"], TEST_PROB["스코어카드"], TEST_PROB["LightGBM 클래스 가중"] = predict_final(test)
results.append(flat("새 모델: 블렌드", f"안정 피처 {len(FINAL_COLS)}개", f"검증에서 고정 {THRESHOLD:.3f}", [test_metrics(TEST_PROB["새 모델: 블렌드"], THRESHOLD)]))
results.append(flat("  구성 모델: 스코어카드", f"안정 피처 {len(FINAL_COLS)}개", f"검증에서 고정 {THRESHOLD_SC:.3f}", [test_metrics(TEST_PROB["스코어카드"], THRESHOLD_SC)]))
results.append(flat("  구성 모델: LightGBM 클래스 가중", f"안정 피처 {len(FINAL_COLS)}개", f"검증에서 고정 {THRESHOLD_LG:.3f}", [test_metrics(TEST_PROB["LightGBM 클래스 가중"], THRESHOLD_LG)]))
predict_strict, _ = fit_blend(train_all, STRICT_COLS)
TEST_PROB["블렌드 (엄격한 과거 기준)"] = predict_strict(test)[0]
results.append(flat("  변형: 엄격한 과거 기준", f"{len(STRICT_COLS)}개", f"검증에서 고정 {THRESHOLD_STRICT:.3f}", [test_metrics(TEST_PROB["블렌드 (엄격한 과거 기준)"], THRESHOLD_STRICT)]))

test_table = pd.DataFrame(results).set_index("모델")
print(f"{time.time() - t0:.0f}초")
display(test_table[["피처", "임계값", "F1", "Precision", "Recall", "PR-AUC", "사후 최적 F1"]])
display(test_table[["F1 35~42", "F1 43~49", "PR-AUC 35~42", "PR-AUC 43~49", "ROC-AUC 43~49"]])

**표 읽기**

- **PR-AUC는 새 모델이 가장 높다**: 0.641 (LightGBM + 팀 그래프 0.606, LightGBM 기본 0.585, 규칙 0.538). 위에서부터 순서대로 조사할 때의 품질은 나아졌다.
- **F1은 새 모델이 가장 낮다**: 0.488. Recall(0.690)은 가장 높지만 Precision은 0.377에 그친다. 검증 구간에서 F1이 최대였던 임계값이 테스트 구간에는 너무 낮았다. 불법 비율은 검증 구간(step 15~34) 16.4%에서 테스트 6.5%로 절반 아래가 됐는데, 새 모델이 불법으로 판정한 비율은 15.4%에서 11.9%로만 줄었다.
- **임계값을 테스트 라벨로 골라도(사후 최적 F1) 0.634까지다.** 규칙(0.709)보다 낮으므로 임계값만의 문제가 아니다. 35~42 구간에서 규칙은 Precision 0.893 / Recall 0.739인데, 새 모델이 같은 Recall을 낼 때의 Precision은 0.627이다(10.1의 그림).
- 구성 모델 단독으로는 스코어카드가 F1 0.574 / PR-AUC 0.638, LightGBM 클래스 가중이 0.443 / 0.632이다. 같은 창의 다른 거래를 보지 않는 엄격한 변형은 0.601 / 0.651이다. 블렌드보다 F1이 높게 나왔지만, 테스트 전에 주 모델로 정한 것은 블렌드다. 여기서 바꾸지 않는다.
- **43~49는 모든 방법이 실패한다.** 새 모델 F1 0.026, PR-AUC 0.048. 이 구간의 순위(ROC-AUC)는 팀의 기본 LightGBM(0.846)이 새 모델(0.756)보다 낫다.

In [ ]:
# 임계값을 다르게 정했을 때 (참고). 새 모델의 주 결과는 위 표의 '검증에서 고정' 값이다.
p_new = TEST_PROB["새 모델: 블렌드"]
th_team = tune_threshold(Y[FOLDS[30]["next"]], P["블렌드"][30][FOLDS[30]["next"]])     # 팀 방식: 1~30 모델의 31~34 예측에서
alt = pd.DataFrame([{"임계값을 정한 방법": n, "값": t, **{k: test_metrics(p_new, t)["35~49"][k] for k in ["F1", "Precision", "Recall"]},
                     "F1 35~42": test_metrics(p_new, t)["35~42"]["F1"], "F1 43~49": test_metrics(p_new, t)["43~49"]["F1"]}
                    for n, t in [("전진 검증 예측 15~34 (주 결과)", THRESHOLD), ("팀 방식: 검증 31~34", th_team), ("0.5", 0.5)]]).set_index("임계값을 정한 방법")
display(alt)

In [ ]:
show = ["규칙: 입력 1개·출력 1개", "결정 트리 깊이 2", "LightGBM 기본", "LightGBM 기본 + 팀 그래프 피처", "새 모델: 블렌드"]
colors = [C_RULE, C_TREE, C_LGBM, C_LGBM, C_NEW]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for ax, metric in zip(axes, ["F1", "PR-AUC"]):
    vals = [test_table.loc[n, metric] for n in show]
    bars = ax.barh(range(len(show)), vals, color=colors, height=0.62)
    for b, vv in zip(bars, vals):
        ax.text(vv + 0.008, b.get_y() + b.get_height() / 2, f"{vv:.3f}", va="center", fontsize=10)
    ax.set_yticks(range(len(show)), show if metric == "F1" else [""] * len(show))
    ax.invert_yaxis()
    ax.set_xlim(0, 0.9)
    ax.set_title(f"테스트 {metric} (35~49)", loc="left", fontsize=11)
    ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG / "m_test_summary.png", dpi=130, bbox_inches="tight")
plt.show()

# time step별 F1
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10.5, 5.4), sharex=True, gridspec_kw={"height_ratios": [3, 1.2], "hspace": 0.08})
thr_of = {"결정 트리 깊이 2": th_tree, "LightGBM 기본": 0.5, "새 모델: 블렌드": THRESHOLD}
for name, color in [("결정 트리 깊이 2", C_TREE), ("LightGBM 기본", C_LGBM), ("새 모델: 블렌드", C_NEW)]:
    f1s = [f1_score(y_te[s_te == s], TEST_PROB[name][s_te == s] >= thr_of[name], zero_division=0) for s in TEST_STEPS]
    ax1.plot(list(TEST_STEPS), f1s, marker="o", ms=4, lw=2, color=color, label=name)
ax2.bar(list(TEST_STEPS), [int(y_te[s_te == s].sum()) for s in TEST_STEPS], color=C_MUTE, width=0.7)
for ax in (ax1, ax2):
    ax.axvline(DARK_MARKET_STEP - 0.5, color="#6b7280", ls="--", lw=1)
ax1.set_ylim(-0.03, 1.03)
ax1.set_ylabel("불법 F1")
ax1.set_title("테스트 구간 time step별 불법 F1 (점선: 43, 다크마켓 폐쇄로 해석되는 시점)", loc="left", fontsize=11)
ax1.legend(frameon=False, loc="lower left", fontsize=9)
ax2.set_ylabel("불법 거래 수")
ax2.set_xlabel("time step")
fig.savefig(FIG / "m_test_by_step.png", dpi=130, bbox_inches="tight")
plt.show()

### 10. 사후 분석 (선택에 쓰지 않음)

여기부터는 테스트 라벨을 써서 결과를 해석한다. 모델은 이미 고정됐고, 아래 내용으로 모델을 바꾸지 않는다.

### 10.1 어떤 거래를 잡고 놓치는가

In [ ]:
shape = np.where(test["cnt_1in1out"] == 1, "입력 1개·출력 1개", np.where(test["cnt_n_out"] == 2, "출력 2개", "그 밖"))
pred_new = p_new >= THRESHOLD
rows = []
for pname, pm in [("35~42", PERIODS["35~42"]), ("43~49", PERIODS["43~49"])]:
    for g in ["입력 1개·출력 1개", "출력 2개", "그 밖"]:
        m = pm & (shape == g)
        ill = m & (y_te == 1)
        rows.append({"기간": pname, "거래 모양": g, "거래 수": int(m.sum()), "불법": int(ill.sum()),
                     "불법 비율": y_te[m].mean() if m.sum() else np.nan,
                     "새 모델이 불법 판정": int((m & pred_new).sum()),
                     "그중 실제 불법 (Precision)": y_te[m & pred_new].mean() if (m & pred_new).sum() else np.nan,
                     "잡은 불법 (Recall)": pred_new[ill].mean() if ill.sum() else np.nan})
shape_table = pd.DataFrame(rows).set_index(["기간", "거래 모양"])
display(shape_table)
cm = pd.crosstab(pd.Series(np.where(y_te == 1, "실제 불법", "실제 합법"), name=""), pd.Series(np.where(pred_new, "불법 판정", "합법 판정"), name="새 모델 (35~49)"))
display(cm)

**왜 PR-AUC는 오르고 F1은 내렸나.** 아래 왼쪽 그림은 35~42 구간의 Precision-Recall 곡선이다. 규칙은 점 하나이고, 모델은 곡선이다. 오른쪽은 새 모델의 임계값에 따른 테스트 F1이다. 그 아래 표는 "입력 1개·출력 1개" 거래 가운데 불법의 비율을 기간별로 센 것이다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.0))
m = PERIODS["35~42"]
for name, color in [("LightGBM 기본", C_LGBM), ("새 모델: 블렌드", C_NEW)]:
    pr, rc, _ = precision_recall_curve(y_te[m], TEST_PROB[name][m])
    ax1.plot(rc, pr, color=color, lw=2, label=name)
rule = TEST_PROB["규칙: 입력 1개·출력 1개"][m] >= 0.5
r_p, r_r = y_te[m][rule].mean(), rule[y_te[m] == 1].mean()
ax1.scatter([r_r], [r_p], s=70, color=C_TREE, zorder=5, label="규칙: 입력 1개·출력 1개")
new_pred = TEST_PROB["새 모델: 블렌드"][m] >= THRESHOLD
ax1.scatter([new_pred[y_te[m] == 1].mean()], [y_te[m][new_pred].mean()], s=70, marker="s", color=C_NEW, zorder=5, label="새 모델의 고정 임계값")
ax1.set_xlabel("Recall")
ax1.set_ylabel("Precision")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1.02)
ax1.set_title("35~42 구간 Precision-Recall", loc="left", fontsize=11)
ax1.legend(frameon=False, fontsize=9, loc="lower left")

grid = np.linspace(0.05, 0.95, 91)
ax2.plot(grid, [f1_score(y_te, p_new >= t) for t in grid], color=C_NEW, lw=2, label="새 모델: 테스트 F1")
ax2.axvline(THRESHOLD, color=C_NEW, ls="--", lw=1)
ax2.annotate(f"검증에서 고정한 값 {THRESHOLD:.2f}", (THRESHOLD, 0.08), textcoords="offset points", xytext=(6, 0), fontsize=9)
ax2.axhline(test_table.loc["규칙: 입력 1개·출력 1개", "F1"], color=C_TREE, lw=1.5, label="규칙의 F1")
ax2.set_xlabel("임계값")
ax2.set_ylabel("불법 F1 (35~49)")
ax2.set_ylim(0, 0.8)
ax2.set_title("임계값에 따른 새 모델의 테스트 F1 (사후 진단)", loc="left", fontsize=11)
ax2.legend(frameon=False, fontsize=9, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "m_test_pr_threshold.png", dpi=130, bbox_inches="tight")
plt.show()

prec_at_rule_recall = float(max(pr[rc >= r_r]))     # 새 모델이 규칙과 같은 Recall을 낼 때의 Precision (35~42)
rows = []
for name, lo, hi in [("학습 1~30", 1, 30), ("검증 31~34", 31, 34), ("테스트 35~42", 35, 42), ("테스트 43~49", 43, 49)]:
    d = lab[(lab["time_step"] >= lo) & (lab["time_step"] <= hi)]
    g1, g2 = d[d["cnt_1in1out"] == 1], d[(d["cnt_1in1out"] == 0) & (d["cnt_n_out"] == 2)]
    rows.append({"기간": name, "불법 비율(전체)": d["y"].mean(), "1입력·1출력 거래 수": len(g1), "그중 불법 비율": g1["y"].mean(),
                 "불법 중 1입력·1출력의 비중": g1["y"].sum() / d["y"].sum(), "출력 2개 거래의 불법 비율": g2["y"].mean()})
purity_table = pd.DataFrame(rows).set_index("기간")
display(purity_table)
print(f"35~42에서 규칙: Precision {r_p:.3f}, Recall {r_r:.3f} | 새 모델이 같은 Recall을 낼 때의 Precision {prec_at_rule_recall:.3f}")

### 10.2 재학습하면 43 이후가 회복되는가 (팀 계획 E8)

운영 상황을 흉내 낸다. step $t$를 채점할 때, $t - d$까지의 라벨이 도착해 있다고 보고 그때까지의 데이터로 같은 설정의 모델을 다시 학습한다. 테스트 구간의 라벨을 학습에 쓰므로 위의 고정 모델 점수와는 **다른 프로토콜**이다. 따로 읽는다.

$$\hat p_t = f_{\le t-d}(\mathbf{x}_t), \qquad d = 1 \text{ (2주 뒤 라벨 도착)},\; d = 3 \text{ (6주 뒤)}$$

In [ ]:
def prequential(delay):
    def one(t):
        predict, _ = fit_blend(lab[lab["time_step"] <= t - delay], FINAL_COLS, seeds=SEEDS)
        d = test[test["time_step"] == t]
        return d.index.values, predict(d)[0]
    out = np.full(len(test), np.nan)
    for idx, p in Parallel(n_jobs=N_JOBS)(delayed(one)(t) for t in TEST_STEPS):
        out[idx] = p
    return out


t0 = time.time()
PREQ = {"고정 모델 (1~34 학습)": p_new, "재학습, 라벨 2주 지연 (d=1)": prequential(1), "재학습, 라벨 6주 지연 (d=3)": prequential(3)}
print(f"{time.time() - t0:.0f}초")
rows = []
for name, p in PREQ.items():
    m = test_metrics(p, THRESHOLD)
    rows.append({"방식": name, **{f"F1 {k}": m[k]["F1"] for k in PERIODS}, **{f"PR-AUC {k}": m[k]["PR-AUC"] for k in PERIODS},
                 "43~49에서 잡은 불법": int(((p >= THRESHOLD) & (y_te == 1) & PERIODS["43~49"]).sum())})
preq_table = pd.DataFrame(rows).set_index("방식")
display(preq_table)

fig, ax = plt.subplots(figsize=(10.5, 3.6))
for (name, p), color in zip(PREQ.items(), [C_NEW, C_TREE, C_LGBM]):
    f1s = [f1_score(y_te[s_te == s], p[s_te == s] >= THRESHOLD, zero_division=0) for s in TEST_STEPS]
    ax.plot(list(TEST_STEPS), f1s, marker="o", ms=4, lw=2, color=color, label=name)
ax.axvline(DARK_MARKET_STEP - 0.5, color="#6b7280", ls="--", lw=1)
ax.set_ylim(-0.03, 1.03)
ax.set_xlabel("time step")
ax.set_ylabel("불법 F1")
ax.set_title("재학습 여부에 따른 time step별 F1", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower left")
fig.savefig(FIG / "m_prequential.png", dpi=130, bbox_inches="tight")
plt.show()

### 10.3 같은 종류의 라벨을 더 모으면 나아지는가

검증 구간에서 본다(학습 1~30의 일부 → 검증 31~34). 테스트는 쓰지 않는다.

- **라벨 양**: 학습 라벨을 무작위로 10% / 25% / 50% / 100%만 쓴다 (3회 반복 평균)
- **기간**: 가장 최근 $k$개 step만 쓴다

In [ ]:
tr30_idx, v_idx = np.flatnonzero(FOLDS[30]["train"]), np.flatnonzero(FOLDS[30]["next"])


def val_ap(train_idx, seed=0):
    predict, _ = fit_blend(dev.iloc[train_idx], FINAL_COLS, seeds=(seed,))
    return average_precision_score(Y[v_idx], predict(dev.iloc[v_idx])[0])


jobs = [("라벨 양", f"{int(fr * 100)}%", r) for fr in (0.1, 0.25, 0.5, 1.0) for r in range(3)] + \
       [("기간", f"최근 {k} step", 0) for k in (4, 8, 16, 30)]


def run_job(kind, label, r):
    if kind == "라벨 양":
        fr = int(label[:-1]) / 100
        idx = np.random.default_rng(r).choice(tr30_idx, size=int(len(tr30_idx) * fr), replace=False)
    else:
        k = int(label.split()[1])
        idx = tr30_idx[STEP[tr30_idx] > 30 - k]
    return kind, label, len(idx), val_ap(idx, seed=r)


lc = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(run_job)(*j) for j in jobs), columns=["종류", "학습 데이터", "학습 거래 수", "검증 PR-AUC"])
lc_table = lc.groupby(["종류", "학습 데이터"], sort=False).agg(**{"학습 거래 수": ("학습 거래 수", "mean"), "검증 PR-AUC": ("검증 PR-AUC", "mean")})
display(lc_table)

### 10.4 모델이 무엇을 보고 판단하나

In [ ]:
sc_pipe = FINAL_MODELS["scorecard"]
kb, lr = sc_pipe.steps[0][1], sc_pipe.steps[1][1]
coef, pos0, rows = lr.coef_.ravel(), 0, []
for j, c in enumerate(FINAL_COLS):
    nb = kb.n_bins_[j]
    w = coef[pos0:pos0 + nb]
    pos0 += nb
    rows.append({"피처": c, "스코어카드 점수 폭": float(w.max() - w.min()), "가장 불법 쪽 구간": int(np.argmax(w)) + 1, "구간 수": int(nb)})
imp = pd.DataFrame(rows).set_index("피처")
gain = np.mean([m.booster_.feature_importance(importance_type="gain") for m in FINAL_MODELS["lgbm"]], axis=0)
imp["LightGBM 중요도 비중"] = gain / gain.sum()
imp["의미"] = [sem.get(c, "") for c in imp.index]
display(imp.sort_values("스코어카드 점수 폭", ascending=False).head(15))
display(imp.sort_values("LightGBM 중요도 비중", ascending=False).head(10)[["LightGBM 중요도 비중", "의미"]])

### 11. 모델 저장과 사용법

In [ ]:
import joblib

MODEL_DIR = BASE / "models"
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump({"scorecard": FINAL_MODELS["scorecard"], "lgbm": FINAL_MODELS["lgbm"], "columns": FINAL_COLS,
             "threshold": THRESHOLD, "affine": AFFINE, "height_agg7": HEIGHT_AGG7, "unstable": sorted(UNSTABLE),
             "note": "설정 B 블렌드. 입력은 build_features()의 출력 + TX + NBR_IN 중 columns에 있는 열."},
            MODEL_DIR / "m_setting_b_blend.joblib")
pd.DataFrame({"txId": test["txId"].values, "time_step": s_te, **{k: v for k, v in TEST_PROB.items()}}).to_csv(
    RES / "m_test_predictions.csv", index=False, encoding="utf-8-sig")
test_table.round(4).to_csv(RES / "m_test_results.csv", encoding="utf-8-sig")
table({k: score(P[k]) for k in base_names + fs_names + model_names + ["블렌드", "블렌드 (엄격한 과거 기준)"]}).round(4).to_csv(RES / "m_validation.csv", encoding="utf-8-sig")
blend_table.round(4).to_csv(RES / "m_blend_validation.csv", encoding="utf-8-sig")
stab.round(4).to_csv(RES / "m_stability.csv", index=False, encoding="utf-8-sig")

summary = {
    "threshold": THRESHOLD, "n_features": len(FINAL_COLS), "n_unstable": len(UNSTABLE),
    "test": test_table.drop(columns=["피처", "임계값"]).round(6).to_dict(orient="index"),
    "val_official": val_table.drop(columns=["임계값"]).round(6).to_dict(orient="index"),
    "validation": {k: {m: round(x, 6) for m, x in score(P[k]).items()} for k in ["블렌드", "스코어카드", "LightGBM 클래스 가중", "LightGBM 기본 (raw)", "규칙: 입력 1개·출력 1개"] + fs_names},
    "prequential": preq_table.round(6).to_dict(orient="index"),
    "learning_curve": {f"{a} {b}": round(float(r["검증 PR-AUC"]), 6) for (a, b), r in lc_table.iterrows()},
    "shape": {f"{a} | {b}": {k: (None if pd.isna(x) else round(float(x), 6)) for k, x in r.items()} for (a, b), r in shape_table.iterrows()},
    "confusion": {f"{i} / {j}": int(cm.loc[i, j]) for i in cm.index for j in cm.columns},
    "alt_threshold": alt.round(6).to_dict(orient="index"),
    "purity": purity_table.round(6).to_dict(orient="index"),
    "rule_35_42": {"precision": round(float(r_p), 4), "recall": round(float(r_r), 4), "new_precision_at_same_recall": round(prec_at_rule_recall, 4)},
    "flag_rate_test": round(float((p_new >= THRESHOLD).mean()), 4),
    "flag_rate_oof": round(float((oof(P["블렌드"])[np.isfinite(oof(P["블렌드"]))] >= THRESHOLD).mean()), 4),
    "prevalence_oof": round(float(Y[np.isfinite(oof(P["블렌드"]))].mean()), 4),
    "runtime_min": round((time.time() - T_START) / 60, 1),
}
(RES / "m_final_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"저장: {MODEL_DIR / 'm_setting_b_blend.joblib'}  |  전체 실행 {summary['runtime_min']}분")

**새 거래에 쓰는 법**

```python
import joblib
bundle = joblib.load("models/m_setting_b_blend.joblib")
feats, _ = build_features(df_all, edges)                    # 이 노트북 1장의 함수. 같은 창의 거래 전체를 함께 넣는다
X = df_all[["txId"] + TX + NBR_IN].merge(feats, on="txId")[bundle["columns"]]
p_sc = bundle["scorecard"].predict_proba(X.fillna(-1.0).to_numpy("float32"))[:, 1]
p_lg = sum(m.predict_proba(X.to_numpy("float32"))[:, 1] for m in bundle["lgbm"]) / len(bundle["lgbm"])
prob = (p_sc + p_lg) / 2
is_illicit = prob >= bundle["threshold"]
```

Elliptic 형식이 아닌 새 데이터라면 `build_features`가 쓰는 원값(입력 수, 출력 수, 금액, 수수료, 선행 거래의 블록 높이와 모양)을 직접 채워 같은 열을 만들면 된다. 모든 열이 실제 단위(BTC, 개수, 블록 수, 시장 중앙값 대비 비율)라서 가능하다.

### 결론

**새 모델을 어디에 쓸 수 있나**

- 점수 순으로 위에서부터 조사하는 용도에는 새 모델이 팀 기준 모델보다 낫다(PR-AUC 0.606 → 0.641).
- 한 번의 판정으로 불법과 합법을 가르는 용도라면, 이 테스트 구간에서는 "입력 1개·출력 1개" 규칙이 더 낫다(F1 0.709 대 0.488).
- 다만 그 규칙도 시기를 탄다. 규칙에 걸리는 거래의 불법 비율은 검증 66.3% → 테스트 35~42 89.3% → 43~49 0.0%로 움직였다. 테스트 35~42는 규칙이 가장 잘 맞는 시기였다.

**처음 질문에 대한 답**

| 질문 | 답 | 근거 |
|---|---|---|
| 피처를 더 나눌 수 있나 | 있다. 의미 기준 네 묶음에 더해, 라벨 없이 "시점에 따라 수준이 변하는가"로 나눴다. 금액·개수·블록 높이는 실제 단위로 되돌렸다 | 1~2장 |
| 가공하면 성능이 오르나 | 검증에서는 먼 미래일수록 오른다. 테스트에서는 PR-AUC가 +0.035 올랐고 F1은 오르지 않았다 | 5장, 9장 |
| 다른 모델이나 블렌더가 도움이 되나 | 거의 안 된다. 모델 종류 사이 차이가 작고, 2단 스태킹은 단순 평균보다 낫지 않다. 블렌딩 이득은 검증 PR-AUC 0.01 안팎이다 | 6~7장 |
| 데이터를 더 모아야 하나 | 같은 시기·같은 종류의 라벨은 이미 충분하다(10%만 써도 0.755, 전부 0.779). 필요한 것은 **최근 라벨**과 **다른 종류의 정보**다 | 10.2~10.3 |
| 43 이후는 회복되나 | 고정 모델로는 안 된다. 재학습하면 43~49에서 잡는 불법이 9건 → 71건(라벨 2주 지연), 47건(6주 지연)으로 늘지만 F1은 0.187에 그친다 | 10.2 |

**검증이 테스트를 맞히지 못한 부분**

- 1~34 안의 전진 검증에서는 새 모델이 규칙보다 PR-AUC와 F1 모두 높았다(F1 평균 0.678 대 0.575). 테스트에서는 F1이 뒤집혔다.
- 팀 공식 검증(31~34)만으로 골랐다면 기본 LightGBM(검증 F1 0.725)을 골랐을 것이고, 그 모델의 테스트 F1은 0.545이다. 어느 검증 방식도 규칙이 테스트에서 가장 높을 것을 알려 주지 않았다.
- 팀 문서의 질문 "설정 B에서 LightGBM이 얕은 트리보다 낮은 원인"에 대한 답은 이렇다. 학습 구간에 대한 과적합만으로는 설명되지 않는다. 검증에서는 모든 예측 거리에서 부스팅 모델이 규칙보다 높았다. 뒤집힌 것은 테스트 구간에서 불법 거래의 구성이 규칙 쪽으로 쏠렸기 때문이다.

**한계**

- 설정 B의 한계는 그대로다. 피처의 표준화는 전체 데이터 기준이고(복호화로 대부분 해소), 라벨은 실체 단위라 거래끼리 독립이 아니다.
- `rel_*`는 같은 창(6~16블록)의 중앙값을 쓴다. 배포할 때는 직전 블록들의 중앙값으로 대신해야 한다. 직전 step 중앙값만 쓴 엄격한 변형의 결과를 9장에 함께 실었다.
- 테스트는 한 구간뿐이다. 임계값이 걸린 지표(F1)는 그 구간의 불법 비율과 구성에 크게 흔들린다. 보고서에는 PR-AUC와 기간별 수치를 함께 싣는 것이 안전하다.

**다음에 해 볼 것**

1. **거래 시점 기준 주소 통계.** 설정 A의 성능 대부분은 주소 통계에서 나오는데, 그 값이 미래 시점 기준이라 설정 B에서 뺐다. 전체 원장(Kondor, Dryad 약 60GB)으로 "거래 시점까지의" 주소 통계를 다시 계산하면 설정 B에 넣을 수 있다. 이번에는 Dryad가 인증 토큰을 요구해 받지 못했다. 파일만 있으면 이 서버(디스크 4.9TB, 메모리 121GB)에서 계산할 수 있다.
2. **재학습 주기와 임계값 재조정**을 운영 규칙으로 넣는다(10.2). 불법 비율이 바뀌면 임계값도 바뀌어야 한다.
3. **규칙을 1차 기준으로 두고 모델 점수로 보강하는 구조**는 이번 테스트를 본 뒤에 떠올린 것이라 여기서 평가하지 않았다. 새 데이터(팀원 C)나 다른 시기 데이터가 생기면 그때 검증한다.

---
# 5. 후속 실험 (`blend_extras.py`)

4단계에서 만든 변수(`P`, `run_cv`, `score`, `STABLE` 등)를 그대로 쓴다. 모두 1~34 안의 전진 검증으로 비교하고, 지금 설정보다 +0.01 이상 나을 때만 바꾼다.

In [ ]:
GAIN_MIN = 0.01
T_EXTRAS = time.time()
import joblib  # noqa: E402

BASE_BLEND = {T: (P["스코어카드"][T] + P["LightGBM 클래스 가중"][T]) / 2 for T in FOLD_CUTS}
BASE_SCORE = score(BASE_BLEND)
SUMMARY = {"base_blend_cv": BASE_SCORE}
print(f"[재현] 블렌드 전진 검증 PR-AUC 평균 {BASE_SCORE['PR-AUC 평균']:.3f} (노트북 0.780)")


def blend_of(p_sc, p_lg):
    return {T: (p_sc[T] + p_lg[T]) / 2 for T in FOLD_CUTS}


def row(name, s, **kw):
    return {"이름": name, **kw, **{k: round(v, 4) for k, v in s.items()}}


# ---------------------------------------------------------------- 1. 하이퍼파라미터 튜닝 (randomized search)
def tuning():
    rng = np.random.default_rng(0)
    space = {"num_leaves": [7, 15, 31, 63], "min_child_samples": [20, 40, 100, 200],
             "reg_lambda": [0.0, 1.0, 5.0, 10.0], "learning_rate": [0.02, 0.03, 0.05, 0.1],
             "n_estimators": [150, 300, 500], "colsample_bytree": [0.3, 0.5, 0.8]}
    current = dict(num_leaves=15, min_child_samples=40, reg_lambda=5.0, learning_rate=0.03, n_estimators=300,
                   colsample_bytree=0.5)
    configs = [current] + [{k: v[rng.integers(len(v))] for k, v in space.items()} for _ in range(30)]
    rows, preds = [], {}
    for i, cfg in enumerate(configs):
        p = run_cv(lgbm(class_weight="balanced", **cfg), STABLE)
        preds[i] = p
        rows.append(row("현재 설정" if i == 0 else f"후보 {i}", score(blend_of(P["스코어카드"], p)),
                        모델="LightGBM", **{k: v for k, v in cfg.items()},
                        단독_PR_AUC=round(score(p)["PR-AUC 평균"], 4)))
    print(f"  LightGBM {len(configs)}개 완료")

    sc_rows, sc_preds = [], {}
    for C in [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]:
        for bins in [5, 8, 12]:
            def make(seed, C=C, bins=bins):
                return make_pipeline(KBinsDiscretizer(n_bins=bins, encode="onehot", strategy="quantile",
                                                      quantile_method="averaged_inverted_cdf"),
                                     LogisticRegression(C=C, max_iter=3000))
            p = run_cv(make, STABLE, seeds=(0,), fill=True)
            sc_preds[(C, bins)] = p
            sc_rows.append(row("현재 설정" if (C, bins) == (0.05, 8) else f"C={C}, 구간 {bins}",
                               score(blend_of(p, P["LightGBM 클래스 가중"])), 모델="스코어카드", C=C, 구간=bins,
                               단독_PR_AUC=round(score(p)["PR-AUC 평균"], 4)))
    print("  스코어카드 18개 완료")

    lg = pd.DataFrame(rows).sort_values("PR-AUC 평균", ascending=False)
    sc = pd.DataFrame(sc_rows).sort_values("PR-AUC 평균", ascending=False)
    best_lg = configs[int(lg.iloc[0]["이름"].split()[-1]) if lg.iloc[0]["이름"] != "현재 설정" else 0]
    best_sc = (sc.iloc[0]["C"], int(sc.iloc[0]["구간"]))
    i_lg = configs.index(best_lg)
    both = score(blend_of(sc_preds[best_sc], preds[i_lg]))
    out = pd.concat([lg, sc], ignore_index=True)
    out.to_csv(RES / "blend_tuning.csv", index=False, encoding="utf-8-sig")
    gain = both["PR-AUC 평균"] - BASE_SCORE["PR-AUC 평균"]
    res = {"best_lgbm": best_lg, "best_scorecard": {"C": best_sc[0], "bins": best_sc[1]},
           "best_both_cv": both, "gain": gain, "adopt": bool(gain >= GAIN_MIN),
           "lgbm_range": [float(lg["PR-AUC 평균"].min()), float(lg["PR-AUC 평균"].max())],
           "sc_range": [float(sc["PR-AUC 평균"].min()), float(sc["PR-AUC 평균"].max())]}
    print(f"[튜닝] 최고 조합 PR-AUC 평균 {both['PR-AUC 평균']:.3f} (현재 {BASE_SCORE['PR-AUC 평균']:.3f}, "
          f"차이 {gain:+.3f}) → {'바꿈' if res['adopt'] else '유지'}")
    print(f"       LightGBM 31개 범위 {res['lgbm_range'][0]:.3f}~{res['lgbm_range'][1]:.3f}, "
          f"스코어카드 18개 범위 {res['sc_range'][0]:.3f}~{res['sc_range'][1]:.3f}")
    return res


# ---------------------------------------------------------------- 2. 불균형 처리 (E5)
class UnderSampled:
    """음성(합법)을 양성 수의 ratio배만 무작위로 남겨 학습한다."""
    def __init__(self, seed, ratio):
        self.seed, self.ratio = seed, ratio
        self.model = lgbm(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                          reg_lambda=5.0)(seed)

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
        keep = np.concatenate([pos, rng.choice(neg, min(len(neg), self.ratio * len(pos)), replace=False)])
        self.model.fit(X[keep], y[keep])
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)


def imbalance():
    base = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=40, reg_lambda=5.0)
    ratio = float((Y == 0).sum() / (Y == 1).sum())
    variants = {
        "클래스 가중 balanced (현재)": P["LightGBM 클래스 가중"],
        "가중 없음": run_cv(lgbm(**base), STABLE),
        f"가중 √(음성/양성) = {ratio ** 0.5:.1f}": run_cv(lgbm(scale_pos_weight=ratio ** 0.5, **base), STABLE),
        "언더샘플링 1:1": run_cv(lambda s: UnderSampled(s, 1), STABLE),
        "언더샘플링 1:3": run_cv(lambda s: UnderSampled(s, 3), STABLE),
    }
    rows = []
    for name, p in variants.items():
        rows.append(row(name, score(blend_of(P["스코어카드"], p)), 단독_PR_AUC=round(score(p)["PR-AUC 평균"], 4)))
    out = pd.DataFrame(rows).sort_values("PR-AUC 평균", ascending=False)
    out.to_csv(RES / "blend_imbalance.csv", index=False, encoding="utf-8-sig")
    best = out.iloc[0]
    gain = best["PR-AUC 평균"] - BASE_SCORE["PR-AUC 평균"]
    print(f"[불균형] 최고 '{best['이름']}' {best['PR-AUC 평균']:.3f} (현재 {BASE_SCORE['PR-AUC 평균']:.3f}, 차이 {gain:+.3f})")
    return {"table": out[["이름", "PR-AUC 평균", "F1 평균", "단독_PR_AUC"]].to_dict(orient="records"),
            "adopt": bool(gain >= GAIN_MIN and best["이름"] != "클래스 가중 balanced (현재)")}


# ---------------------------------------------------------------- 3. 임계값 방식
K_GRID = np.round(np.arange(0.005, 0.30, 0.005), 3)


def flag_top_k(prob, steps, k):
    """time step마다 점수 상위 k 비율을 경보로 낸다. 같은 step 안의 순위만 쓴다."""
    out = np.zeros(len(prob), bool)
    for t in np.unique(steps):
        m = steps == t
        n = max(1, int(round(k * m.sum())))
        idx = np.flatnonzero(m)[np.argsort(-prob[m])[:n]]
        out[idx] = True
    return out


def choose_k(ys, ps, ss):
    f1s = [f1_score(ys, flag_top_k(ps, ss, k), zero_division=0) for k in K_GRID]
    return float(K_GRID[int(np.argmax(f1s))])


def threshold_methods(preds):
    """세 방식의 F1을 예측 거리별로 잰다. 기준값은 모두 그보다 과거 폴드에서만 정한다."""
    res = {"확률 임계값 (현재)": [], "time step별 상위 k%": [], "고정 경보율 (과거 불법 비율)": []}
    for lo, hi in BUCKETS:
        acc = {k: [] for k in res}
        for T in FOLD_CUTS:
            m = (STEP >= T + lo) & (STEP <= min(T + hi, 34))
            past = [t for t in FOLD_CUTS if t < T]
            if not past or m.sum() == 0 or Y[m].sum() == 0:
                continue
            pm = np.concatenate([FOLDS[t]["next"].nonzero()[0] for t in past])
            th = tune_threshold(Y[pm], np.concatenate([preds[t][FOLDS[t]["next"]] for t in past]))
            acc["확률 임계값 (현재)"].append(f1_score(Y[m], preds[T][m] >= th, zero_division=0))
            ys = np.concatenate([Y[FOLDS[t]["next"]] for t in past])
            ps = np.concatenate([preds[t][FOLDS[t]["next"]] for t in past])
            ss = np.concatenate([STEP[FOLDS[t]["next"]] for t in past])
            k = choose_k(ys, ps, ss)
            acc["time step별 상위 k%"].append(f1_score(Y[m], flag_top_k(preds[T][m], STEP[m], k), zero_division=0))
            rate = float(Y[STEP <= T].mean())     # 학습 구간의 불법 비율만큼 경보
            acc["고정 경보율 (과거 불법 비율)"].append(
                f1_score(Y[m], flag_top_k(preds[T][m], STEP[m], rate), zero_division=0))
        for k in res:
            res[k].append(float(np.mean(acc[k])))
    rows = [{"방식": k, "F1 h1~4": v[0], "F1 h5~8": v[1], "F1 h9~15": v[2], "F1 평균": float(np.mean(v))}
            for k, v in res.items()]
    out = pd.DataFrame(rows).round(4)
    out.to_csv(RES / "blend_threshold.csv", index=False, encoding="utf-8-sig")
    base = out.loc[out["방식"] == "확률 임계값 (현재)", "F1 평균"].item()
    best = out.sort_values("F1 평균", ascending=False).iloc[0]
    print(f"[임계값] 최고 '{best['방식']}' F1 평균 {best['F1 평균']:.3f} (현재 {base:.3f})")
    return {"table": out.to_dict(orient="records"), "best": best["방식"],
            "adopt": bool(best["방식"] != "확률 임계값 (현재)" and best["F1 평균"] - base >= GAIN_MIN)}


# ---------------------------------------------------------------- 4. 반지도: self-training (E7)
def self_training(hi=0.9, lo=0.02):
    unl = df_all.loc[~is_lab, ["txId", "time_step"] + RAW].merge(feats, on="txId").reset_index(drop=True)
    U_step = unl["time_step"].values
    Xd, Xdf = dev[STABLE].to_numpy(np.float32), dev_f[STABLE].to_numpy(np.float32)
    Xu, Xuf = unl[STABLE].to_numpy(np.float32), unl[STABLE].fillna(-1.0).to_numpy(np.float32)
    preds, added = {}, {}

    def fit_pred(Xa, Xaf, ya, ev):
        sc = make_scorecard().fit(Xaf, ya)
        lgs = [LGBM_CW(s).fit(Xa, ya) for s in SEEDS]

        def predict(X, Xf):
            return (sc.predict_proba(Xf)[:, 1] + np.mean([m.predict_proba(X)[:, 1] for m in lgs], axis=0)) / 2
        return predict

    for T in FOLD_CUTS:
        tr, ev = FOLDS[T]["train"], FOLDS[T]["long"]
        um = U_step <= T                                   # 학습 구간과 같은 시기의 라벨 없는 거래만
        pred0 = fit_pred(Xd[tr], Xdf[tr], Y[tr], ev)
        pu = pred0(Xu[um], Xuf[um])
        pos, neg = pu >= hi, pu <= lo
        Xa = np.vstack([Xd[tr], Xu[um][pos | neg]])
        Xaf = np.vstack([Xdf[tr], Xuf[um][pos | neg]])
        ya = np.concatenate([Y[tr], pos[pos | neg].astype(int)])
        pred1 = fit_pred(Xa, Xaf, ya, ev)
        p = np.full(len(dev), np.nan)
        p[ev] = pred1(Xd[ev], Xdf[ev])
        preds[T] = p
        added[T] = {"가짜 불법": int(pos.sum()), "가짜 합법": int(neg.sum()), "라벨": int(tr.sum())}
    s = score(preds)
    gain = s["PR-AUC 평균"] - BASE_SCORE["PR-AUC 평균"]
    pd.DataFrame([row("블렌드 (현재)", BASE_SCORE), row(f"self-training (≥{hi} 불법, ≤{lo} 합법)", s)]).to_csv(
        RES / "blend_selftrain.csv", index=False, encoding="utf-8-sig")
    print(f"[반지도] self-training PR-AUC 평균 {s['PR-AUC 평균']:.3f} (현재 {BASE_SCORE['PR-AUC 평균']:.3f}, 차이 {gain:+.3f})")
    return {"cv": s, "gain": gain, "added": added, "adopt": bool(gain >= GAIN_MIN)}


# ---------------------------------------------------------------- 5. 해석(SHAP)과 오류 분석 (최종 모델, 선택에 쓰지 않음)
def explain_and_errors():
    saved = joblib.load(BASE / "models" / "m_setting_b_blend.joblib")
    cols, thr = saved["columns"], saved["threshold"]
    lab_all = df_all.loc[is_lab, ["txId", "time_step", "y"] + RAW].merge(feats, on="txId")
    test = lab_all[lab_all["time_step"] >= 35].reset_index(drop=True)
    X = test[cols].to_numpy(np.float32)
    # LightGBM 내장 TreeSHAP (pred_contrib): 마지막 열은 기준값
    contrib = np.mean([m.booster_.predict(X, pred_contrib=True)[:, :-1] for m in saved["lgbm"]], axis=0)
    early = test["time_step"].values < DARK_MARKET_STEP
    sem = pd.read_csv(BASE / "feature_semantics.csv").set_index("column")["description"]
    imp = pd.DataFrame({"피처": cols,
                        "평균 |SHAP| 35~49": np.abs(contrib).mean(0),
                        "평균 |SHAP| 35~42": np.abs(contrib[early]).mean(0),
                        "평균 |SHAP| 43~49": np.abs(contrib[~early]).mean(0),
                        "불법 쪽 평균 SHAP (불법 거래)": contrib[test["y"].values == 1].mean(0)})
    imp["의미"] = [sem.get(c, "") for c in cols]
    imp = imp.sort_values("평균 |SHAP| 35~49", ascending=False).reset_index(drop=True)
    imp.round(4).to_csv(RES / "blend_shap_top.csv", index=False, encoding="utf-8-sig")

    top = imp.head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    y = np.arange(len(top))
    ax.barh(y + 0.2, top["평균 |SHAP| 35~42"], 0.4, color="#2a78d6", label="35~42 (폐쇄 전)")
    ax.barh(y - 0.2, top["평균 |SHAP| 43~49"], 0.4, color="#eb6834", label="43~49 (폐쇄 후)")
    ax.set_yticks(y, top["피처"])
    ax.set_xlabel("평균 |SHAP| (LightGBM 쪽, 로그 오즈)")
    ax.set_title("최종 블렌드의 LightGBM이 크게 본 피처 15개 (테스트 거래)", loc="left", fontweight="bold")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG / "blend_shap.png", dpi=130, bbox_inches="tight")
    plt.close(fig)

    # 오류 분석: 노트북이 저장한 최종 예측과 고정 임계값
    pred = pd.read_csv(RES / "m_test_predictions.csv", encoding="utf-8-sig")
    d = test[["txId", "time_step", "y", "cnt_n_in", "cnt_n_out"]].merge(pred[["txId", "새 모델: 블렌드"]], on="txId")
    d["경보"] = d["새 모델: 블렌드"] >= thr
    d["기간"] = np.where(d["time_step"] < DARK_MARKET_STEP, "35~42", "43~49")
    n_in, n_out = d["cnt_n_in"], d["cnt_n_out"]
    d["모양"] = np.select([(n_in == 1) & (n_out == 1), (n_in == 1) & (n_out == 2), (n_in >= 3) & (n_out == 1),
                         n_out >= 3], ["입력1·출력1", "입력1·출력2", "여러 입력→출력1", "출력 3개 이상"], "기타")
    d["결과"] = np.select([d["y"].eq(1) & d["경보"], d["y"].eq(1) & ~d["경보"], d["y"].eq(0) & d["경보"]],
                        ["TP (잡음)", "FN (놓침)", "FP (오탐)"], "TN")
    cm = d.pivot_table(index="기간", columns="결과", values="txId", aggfunc="count", fill_value=0)
    by_shape = d[d["결과"] != "TN"].pivot_table(index=["기간", "모양"], columns="결과", values="txId",
                                                aggfunc="count", fill_value=0)
    pd.concat({"혼동 행렬": cm}, names=["표"]).to_csv(RES / "blend_errors_confusion.csv", encoding="utf-8-sig")
    by_shape.to_csv(RES / "blend_errors_by_shape.csv", encoding="utf-8-sig")
    print("[오류] 혼동 행렬\n", cm.to_string())
    print("[오류] 모양별\n", by_shape.to_string())
    print("[해석] SHAP 상위 10\n", imp.head(10)[["피처", "평균 |SHAP| 35~42", "평균 |SHAP| 43~49", "의미"]].to_string())
    return {"confusion": cm.to_dict(), "by_shape": {f"{a} | {b}": r.to_dict() for (a, b), r in by_shape.iterrows()},
            "shap_top10": imp.head(10)[["피처", "평균 |SHAP| 35~49", "평균 |SHAP| 35~42", "평균 |SHAP| 43~49", "의미"]]
            .round(4).to_dict(orient="records")}

In [ ]:
SUMMARY["tuning"] = tuning()
SUMMARY["imbalance"] = imbalance()
SUMMARY["threshold"] = threshold_methods(BASE_BLEND)
SUMMARY["self_training"] = self_training()
SUMMARY["explain"] = explain_and_errors()
SUMMARY["any_change"] = any(SUMMARY[k]["adopt"] for k in ["tuning", "imbalance", "threshold", "self_training"])
SUMMARY["runtime_min"] = round((time.time() - T_EXTRAS) / 60, 1)
(RES / "blend_extras_summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2, default=float),
                                               encoding="utf-8")
print(f"\n최종 모델을 바꿀 결과가 있나: {SUMMARY['any_change']}  |  {SUMMARY['runtime_min']}분")

In [ ]:
cv_cols = ["이름", "PR-AUC h1~4", "PR-AUC h5~8", "PR-AUC h9~15", "PR-AUC 평균", "F1 평균", "단독_PR_AUC"]
print("① 튜닝 (상위 10개)"); t = show_csv("results/blend_tuning.csv", 10)
print("② 불균형 처리"); show_csv("results/blend_imbalance.csv", cols=cv_cols)
print("③ 임계값 방식"); show_csv("results/blend_threshold.csv")
print("④ self-training"); show_csv("results/blend_selftrain.csv")
print("⑤ SHAP 상위 15"); show_csv("results/blend_shap_top.csv", 15)
print("⑥ 오류 분석"); show_csv("results/blend_errors_confusion.csv"); show_csv("results/blend_errors_by_shape.csv")
display(Image(ROOT / "figures" / "blend_shap.png"))

---
# 6. 튜닝 블렌드 테스트 채점 (`blend_tuned_test.py`)

5단계 튜닝 결과(`results/blend_extras_summary.json`)의 설정을 1~34로 학습하고 35~49를 채점한다. 팀에서 이미 한 번 채점한 결과(PR-AUC 0.640, F1 0.477)를 재현하는 단계다. **이 결과를 보고 설정을 바꾸지 않는다.**

In [ ]:
summary = json.loads((RES / "blend_extras_summary.json").read_text(encoding="utf-8"))
LG_CFG = summary["tuning"]["best_lgbm"]
SC_CFG = summary["tuning"]["best_scorecard"]
LG_CFG = {k: (int(v) if k in ("num_leaves", "min_child_samples", "n_estimators") else float(v)) for k, v in LG_CFG.items()}
make_lg = lgbm(class_weight="balanced", **LG_CFG)


def make_sc(seed=0):
    return make_pipeline(KBinsDiscretizer(n_bins=int(SC_CFG["bins"]), encode="onehot", strategy="quantile",
                                          quantile_method="averaged_inverted_cdf"),
                         LogisticRegression(C=float(SC_CFG["C"]), max_iter=3000))


# 1) 임계값: 1~34 안의 전진 검증 예측으로만
cv = {T: (a + b) / 2 for T, a, b in zip(FOLD_CUTS, run_cv(make_sc, STABLE, seeds=(0,), fill=True).values(),
                                         run_cv(make_lg, STABLE).values())}
cv_score = score(cv)
o = oof(cv)
m = np.isfinite(o)
THR = tune_threshold(Y[m], o[m])
print(f"튜닝 블렌드 전진 검증 PR-AUC 평균 {cv_score['PR-AUC 평균']:.3f}, 임계값 {THR:.3f}")

# 2) 1~34 전체로 학습 → 35~49 한 번 채점
lab = df_all.loc[is_lab, ["txId", "time_step", "y"] + RAW].merge(feats, on="txId")
lab["y"] = lab["y"].astype(int)
train, test = lab[lab["time_step"] <= 34], lab[lab["time_step"] >= 35].reset_index(drop=True)
sc = make_sc().fit(train[STABLE].fillna(-1.0).to_numpy(np.float32), train["y"].values)
lgs = [make_lg(s).fit(train[STABLE].to_numpy(np.float32), train["y"].values) for s in FINAL_SEEDS]
p_sc = sc.predict_proba(test[STABLE].fillna(-1.0).to_numpy(np.float32))[:, 1]
p_lg = np.mean([g.predict_proba(test[STABLE].to_numpy(np.float32))[:, 1] for g in lgs], axis=0)
prob = (p_sc + p_lg) / 2

y, step = test["y"].values, test["time_step"].values
first = pd.read_csv(RES / "m_test_predictions.csv", encoding="utf-8-sig").set_index("txId").loc[test["txId"], "새 모델: 블렌드"].values
first_thr = joblib.load(BASE / "models" / "m_setting_b_blend.joblib")["threshold"]
rows = []
for name, p, th in [("블렌드 (첫 채점, 노트북 설정)", first, first_thr), ("튜닝 블렌드 (두 번째 채점)", prob, THR)]:
    for period, mask in [("35~49", np.ones(len(y), bool)), ("35~42", step < DARK_MARKET_STEP), ("43~49", step >= DARK_MARKET_STEP)]:
        e = evaluate(y[mask], p[mask], th)
        rows.append({"모델": name, "기간": period, "임계값": round(th, 3), "PR-AUC": e["pr_auc"], "F1": e["f1"],
                     "Precision": e["precision"], "Recall": e["recall"], "ROC-AUC": roc_auc_score(y[mask], p[mask]),
                     "잡은 불법": int(((p[mask] >= th) & (y[mask] == 1)).sum())})
out = pd.DataFrame(rows).round(4)
out.to_csv(RES / "blend_tuned_test.csv", index=False, encoding="utf-8-sig")
print(out.to_string(index=False))

joblib.dump({"scorecard": sc, "lgbm": lgs, "columns": STABLE, "threshold": THR, "scorecard_cfg": SC_CFG,
             "lgbm_cfg": LG_CFG, "cv": cv_score}, BASE / "models" / "m_setting_b_blend_tuned.joblib")

---
# 7. 요약

In [ ]:
ex = json.loads((ROOT / "results" / "blend_extras_summary.json").read_text(encoding="utf-8"))
tt = pd.read_csv(ROOT / "results" / "blend_tuned_test.csv", encoding="utf-8-sig")
fin = tt[(tt["모델"].str.contains("튜닝")) & (tt["기간"] == "35~49")].iloc[0]
print(f"블렌드 전진 검증 PR-AUC 평균      {ex['base_blend_cv']['PR-AUC 평균']:.3f}")
print(f"튜닝 후 전진 검증 PR-AUC 평균     {ex['tuning']['best_both_cv']['PR-AUC 평균']:.3f}  (바꿈: {ex['tuning']['adopt']})")
print(f"불균형·임계값·self-training 중 바꿀 것  {[k for k in ['imbalance', 'threshold', 'self_training'] if ex[k]['adopt']] or '없음'}")
print(f"최종 모델 테스트  PR-AUC {fin['PR-AUC']:.3f}  F1 {fin['F1']:.3f}  (팀 기준 0.640 / 0.477)")